<p class="lead">Everything in this course so far produced evidence. This unit turns evidence into a document other people can judge: figures drawn at their final size in code, tables whose digits mean what they appear to mean, a Methods section detailed enough for a stranger to reproduce the result, the reporting checklists reviewers in our fields actually apply, and an honest account of how the text and the code were written — including the part written with a language model. The unit is deliberately mechanical: almost every object in a paper can be generated from the analysis itself, and the ones that are generated are the ones that stay correct when a number changes.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata la última etapa de un proyecto: convertir resultados en un manuscrito defendible. Se construyen figuras de calidad de publicación íntegramente con código R (tamaño en centímetros, resolución, exportación a PDF y SVG con tipografías incrustadas mediante <code>cairo_pdf</code>, un tema reutilizable de <code>ggplot2</code>, composición de paneles con <code>patchwork</code> y letras de panel, comprobaciones de daltonismo y de escala de grises) y se rehace una figura exploratoria del curso como figura de dos paneles. Se diseñan tablas con cifras significativas coherentes con su incertidumbre, exportadas con <code>knitr::kable</code> a LaTeX y a Markdown, y suplementos legibles por máquina con su diccionario de datos. Se redacta la sección de Métodos del modelo de ROCK1 a partir de la procedencia que el propio documento captura, y se completan las listas de verificación que exige la literatura del grupo: principios de validación de la OCDE y formato QMRF, TRIPOD+AI, MIAME/MINSEQE y las buenas prácticas de LiveCoMS. Finalmente se tratan las declaraciones de disponibilidad de datos y código, las licencias, la estructura narrativa del artículo y el uso declarado y verificado de modelos de lenguaje, con una comprobación automática de la integridad de las referencias contra Crossref escrita con <code>httr2</code>.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Produce publication-quality figures programmatically in R (vector formats, physical sizes in centimetres, embedded fonts, multi-panel layouts, colour) that pass journal requirements without manual editing.</li>
<li>Design tables and supplementary material that a reader can reuse, and write Methods that allow replication (package versions, parameters, seeds, data checksums).</li>
<li>Apply the reporting checklists relevant to the laboratory — OECD principles and QMRF, TRIPOD+AI for clinical prediction, MIAME/MINSEQE for omics, LiveCoMS best practices for simulation — and the journal's data and code policies.</li>
<li>Use large language models in writing and coding ethically and transparently, and detect the failure mode that matters most: fabricated or mismatched references.</li>
<li>Avoid the common writing pitfalls of computational papers: over-claiming, metric cherry-picking, uncredited data and figures that mislead.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Run every chunk from top to bottom; every number quoted in the prose is printed by the chunk above or below it. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","ranger","patchwork","httr2","digest"))</code> once before the setup chunk. One chunk in Section 15 queries the public Crossref API through <code>httr2</code>; it is wrapped in <code>tryCatch()</code> and falls back to a frozen copy of the responses, and the document prints which path it took. Everything else runs offline from <code>../data/</code>. The model whose results are reported throughout is refitted here in about ten seconds, so the Methods section and the checklists describe a model that exists rather than one quoted from elsewhere.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(ranger); library(ggplot2)
  library(patchwork);  library(knitr);  library(httr2);  library(digest)
})
set.seed(0)
PAL  <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
OKABE_ITO <- c("#0072B2", "#D55E00", "#009E73", "#CC79A7",
               "#E69F00", "#56B4E9", "#F0E442", "#000000")
CM   <- 1 / 2.54                       ## graphics devices think in inches, journals in centimetres
WORK <- file.path(tempdir(), "u23"); dir.create(WORK, showWarnings = FALSE)
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp   <- function(f) data_file(f)
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ranger", as.character(packageVersion("ranger")),
    "| ggplot2", as.character(packageVersion("ggplot2")),
    "| patchwork", as.character(packageVersion("patchwork")),
    "| knitr", as.character(packageVersion("knitr")),
    "| httr2", as.character(packageVersion("httr2")), "\n")
cat("scratch folder:", basename(WORK), "inside the session's tempdir()\n")



## 1. What the last mile is for

A paper is not a report of what you did; it is an argument that a specific claim is supported by specific evidence, written so that a sceptical reader can check it. Three audiences read it, in this order: an editor deciding whether to send it out, two or three referees deciding whether the evidence supports the claim, and — if it survives — the student in another laboratory who wants to reuse the method two years later. Each of them reads different parts. The editor reads the title, the abstract and the figures. The referee reads the figures, then the Methods, then the parts of the Results that the figures made them suspicious about. The reuser reads the Methods and the data-availability statement and nothing else.

This ordering explains the advice the laboratory repeats to every student: **assemble the figures first, then write the story around them**<sup class="c"><a href="#r9">[9]</a></sup>. A figure that cannot be explained in two sentences is usually an analysis that has not finished, and writing the captions before the Results forces every claim to point at visible evidence.

The second reason the last mile deserves a whole unit is that almost all of it can be *computed*. A figure drawn by code can be redrawn when the model is refitted; a figure assembled by hand in a drawing program cannot, and so it quietly stops matching the numbers. A Methods paragraph generated from the session that ran the analysis cannot claim the wrong version of `ranger`. Reproducibility failures in computational papers are overwhelmingly bookkeeping failures, not conceptual ones<sup class="c"><a href="#r14">[14]</a><a href="#r20">[20]</a><a href="#r29">[29]</a></sup>, and bookkeeping is what a knitted document is good at. This unit is itself the demonstration: the HTML page you are reading was produced by `knitr` and R Markdown<sup class="c"><a href="#r35">[35]</a></sup> from the source file, so no number in it can be stale.

<div class="box bridge"><h4>Domain context — where the group publishes, and what its referees check</h4>
<p>The laboratory works at the intersection of computational chemistry, bioinformatics and machine learning for drug discovery, and publishes in journals that are Q1 or Q2 and charge no article-processing fee to the author. In practice that means outlets of the <em>Journal of Chemical Information and Modeling</em>, <em>Journal of Cheminformatics</em>, <em>Bioinformatics</em>, <em>Briefings in Bioinformatics</em>, <em>Molecules</em>, <em>International Journal of Molecular Sciences</em>, <em>Pharmaceuticals</em> and <em>Frontiers</em> type. A <strong>QSAR</strong> paper (quantitative structure–activity relationship: a model that predicts a biological activity from the chemical structure of a compound) submitted to such a journal is checked by referees for a short list of things, almost always the same list: is the data set described well enough to be rebuilt, including the target, the assay types and the curation; is the validation external rather than only internal; is the <strong>applicability domain</strong> — the region of chemical space where the model may be trusted — defined; are the performance metrics reported on the split that matches the claim; are the compounds and the code available. These are not matters of taste. They are the five <strong>OECD principles</strong> for the validation of (Q)SAR models<sup class="c"><a href="#r12">[12]</a><a href="#r13">[13]</a></sup>, and Section 12 fills them in for the model this document fits.</p></div>

A thesis and a paper contain the same evidence in different documents. The thesis shows the path, including the failed featurisation and the split that leaked; the paper shows the destination and the evidence that it is the right one. The discipline this unit teaches is to decide, for every object you produce, whether it belongs in the paper, in the supplement, in the repository or in the thesis only.



In [ ]:
df  <- fread(dp("rock1_chembl_curated.csv"))
ex  <- df[censored == FALSE]            ## the regression target is defined only for exact values
DESC <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
          "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
rep_ <- ex[n_measurements > 1]
sigma_exp <- sqrt(sum((rep_$n_measurements - 1) * rep_$pActivity_sd^2, na.rm = TRUE) /
                  sum(rep_$n_measurements - 1, na.rm = TRUE))
r2_ceiling <- 1 - sigma_exp^2 / var(ex$pActivity)
raw_sha <- digest(dp("rock1_chembl_curated.csv"), algo = "sha256", file = TRUE)
cat(sprintf("curated table: %d compounds x %d columns; %d with at least one exact measurement, %d censored\n",
            nrow(df), ncol(df), nrow(ex), sum(df$censored)))
cat(sprintf("replicates: %d compounds measured more than once; pooled experimental sd sigma_exp = %.3f log units -> R2 ceiling about %.2f\n",
            nrow(rep_), sigma_exp, r2_ceiling))
cat("SHA-256 of the curated file:", substr(raw_sha, 1, 32), "...\n")



The pooled standard deviation printed above is the number against which every performance metric in this unit must be read. It is the spread between repeated measurements of the *same* compound in the public literature, so no model — however good — can predict this data set with a root-mean-square error much below it<sup class="c"><a href="#r30">[30]</a></sup>. Unit 6 derived it in detail; it is recomputed here because a paper may not quote a number it did not produce.

Now the model. It is the running model of the course, built on the ChEMBL snapshot described in Unit 3<sup class="c"><a href="#r26">[26]</a></sup>: Morgan fingerprints (radius 2, 2,048 bits) concatenated with eleven RDKit descriptors, a random forest<sup class="c"><a href="#r27">[27]</a></sup> of 200 trees fitted with `ranger`<sup class="c"><a href="#r33">[33]</a></sup>, evaluated twice — under a random five-fold cross-validation and under a Bemis–Murcko scaffold hold-out<sup class="c"><a href="#r31">[31]</a></sup> in which no scaffold appears on both sides of the split. Reporting both is the whole point: the first number is what an optimistic paper quotes, the second is what the model will do on new chemistry.

<div class="box bridge"><h4>Substitution — where the fingerprints come from</h4>
<p>R has no RDKit binding in this environment, so the Morgan fingerprints are not recomputed here: they are read from the frozen sparse file <code>data/rock1_morgan_sparse.csv.gz</code> (two columns, <code>id</code> = InChIKey and <code>bit</code> = 0-based set bit) and rebuilt with <code>Matrix::sparseMatrix()</code>. <a href="../../data/manifest_fingerprints.json">manifest_fingerprints.json</a> records the RDKit call and the SHA-256 of the file, and <code>data/rock1_curated.sdf.gz</code> ships the structures so that a reader with ChemmineR or rcdk can recompute them. Everything else in this unit is computed in R from the data.</p></div>



In [ ]:
sp   <- fread(dp("rock1_morgan_sparse.csv.gz")); keep <- sp[id %in% ex$inchikey]
FP   <- sparseMatrix(i = match(keep$id, ex$inchikey), j = keep$bit + 1L, x = 1,
                     dims = c(nrow(ex), 2048))
X    <- cbind(as.matrix(FP), as.matrix(ex[, ..DESC]))
colnames(X) <- c(paste0("b", 0:2047), DESC); storage.mode(X) <- "double"
y    <- ex$pActivity; scaf <- ex$scaffold_smiles
HP   <- list(num.trees = 200, seed = 7, num.threads = 4)   ## every hyperparameter of the reported model
## seed = 0 would mean "no seed" to ranger and the fit would not be repeatable; Unit 26 measures that
set.seed(0); folds <- sample(rep(1:5, length.out = nrow(X)))
pred_cv <- numeric(nrow(X))
for (k in 1:5) {
  tr <- folds != k
  m  <- ranger(x = X[tr, ], y = y[tr], num.trees = HP$num.trees, seed = HP$seed, num.threads = HP$num.threads)
  pred_cv[!tr] <- predict(m, X[!tr, ])$predictions
}
rmse_cv <- sqrt(mean((pred_cv - y)^2)); r2_cv <- 1 - sum((pred_cv - y)^2) / sum((y - mean(y))^2)
grp <- split(seq_along(scaf), scaf); ord <- order(-lengths(grp), names(grp))   ## largest families to training
n_tr <- floor(0.8 * length(scaf)); TR <- integer(0); TE <- integer(0)
for (k in ord) { i <- grp[[k]]; if (length(TR) < n_tr) TR <- c(TR, i) else TE <- c(TE, i) }
fit_sc  <- ranger(x = X[TR, ], y = y[TR], num.trees = HP$num.trees, seed = HP$seed, num.threads = HP$num.threads)
pred_te <- predict(fit_sc, X[TE, ])$predictions; yt <- y[TE]
rmse_sc <- sqrt(mean((pred_te - yt)^2)); mae_sc <- mean(abs(pred_te - yt))
r2_sc   <- 1 - sum((pred_te - yt)^2) / sum((yt - mean(yt))^2)
cat(sprintf("random 5-fold CV  : RMSE %.3f   R2 %.3f   (n = %d)\n", rmse_cv, r2_cv, length(y)))
cat(sprintf("scaffold hold-out : RMSE %.3f   MAE %.3f   R2 %.3f  (train %d / test %d, shared scaffolds %d)\n",
            rmse_sc, mae_sc, r2_sc, length(TR), length(TE), length(intersect(scaf[TR], scaf[TE]))))
cat(sprintf("experimental floor: sigma_exp %.3f\n", sigma_exp))



Two numbers, one model, a difference of roughly a seventh of a log unit in RMSE and of a quarter in \( R^{2} \). Which one goes in the abstract is the most consequential writing decision in a QSAR paper, and the answer is: both, with the claim attached to the pessimistic one. A paper quoting only the random-split figure answers a question nobody asked — how well the model interpolates among chemotypes it has already seen.

## 2. The figure as a physical object

A figure in a manuscript is printed at a fixed physical width: one column (about 8.5 cm in most chemistry and biology journals), one and a half columns (about 11.4 cm) or the full page width (about 17.4 cm). Everything else follows from that number. If you draw a figure at the default 7 × 7 inches of R's `pdf()` device and the journal shrinks it to 8.5 cm, every font in it shrinks by a factor of 0.48, and a 10-point axis label arrives at the reader as under 5 points — below the minimum of 5–7 points that most publishers specify. The fix is not to enlarge the fonts afterwards; it is to **draw the figure at its final size from the start** and never scale it again<sup class="c"><a href="#r1">[1]</a></sup>. `ggsave()` makes this natural, because it accepts `units = "cm"`.

The second physical property is resolution. Raster formats (PNG, TIFF) store pixels and must be generated at the publisher's dpi — typically 300 dpi for photographs and 600–1200 dpi for line art. Vector formats (PDF, SVG, EPS) store drawing instructions and are resolution-independent: a scatter plot, a line chart or a bar chart should always be vector. The exception is a figure with tens of thousands of marks — a scatter of 100,000 docking poses will produce a 40 MB PDF that crashes the typesetter — where rasterising only the data layer inside an otherwise vector figure is the professional compromise.

The third is fonts. A PDF that references a font without embedding it renders differently on the publisher's machine, and this is where R differs from Python in an important way: the default `pdf()` device does *not* embed the base-14 fonts, while `cairo_pdf()` does. Several publishers reject the first and accept the second, so the laboratory exports with `device = cairo_pdf` and checks the file. The chunk below writes the same figure five ways and measures the consequences.



In [ ]:
p_demo <- ggplot(ex, aes(MolWt, pActivity)) +
  geom_point(size = 0.4, alpha = 0.25, colour = PAL[["blue"]]) +
  labs(x = "Molecular weight (Da)", y = expression(pActivity~(-log[10]~M)), title = "ROCK1 compounds")
outs <- list(c("fig_col_150.png", "png", "150"), c("fig_col_300.png", "png", "300"),
             c("fig_col_base.pdf", "pdf", NA), c("fig_col_cairo.pdf", "cairo_pdf", NA),
             c("fig_col.svg", "svg", NA))
rows <- lapply(outs, function(o) {
  dev <- switch(o[2], png = ragg::agg_png, pdf = grDevices::pdf,
                cairo_pdf = grDevices::cairo_pdf, svg = grDevices::svg)
  args <- list(filename = file.path(WORK, o[1]), plot = p_demo, width = 8.5, height = 6.0,
               units = "cm", device = dev)
  if (!is.na(o[3])) args$dpi <- as.numeric(o[3])
  do.call(ggsave, args)
  f <- file.path(WORK, o[1]); r <- readBin(f, "raw", file.size(f))
  data.frame(file = o[1], kB = round(file.size(f) / 1024, 1),
             fonts_embedded = length(grepRaw("/FontFile", r, fixed = TRUE)) > 0)
})
print(do.call(rbind, rows), row.names = FALSE)
cat(sprintf("\none-column width 8.5 cm = %.2f in; full width 17.4 cm = %.2f in\n", 8.5 * CM, 17.4 * CM))
cat(sprintf("a 9 pt label in a figure drawn at 17.4 cm and printed at 8.5 cm arrives as %.1f pt\n", 9 * 8.5 / 17.4))



Read the three columns together rather than one at a time. The PDFs stay sharp at any magnification because they store roughly two thousand circle instructions rather than a grid of pixels, and they are smaller than the 300 dpi PNG that the journal would demand; the 150 dpi PNG is the smallest file of all, and it is also the only one that visibly blurs when the reader zooms, which is the trade being made. The SVG is by far the largest here, because every one of the 2,764 points becomes an XML element — it is a format for sending a co-author a figure whose text they must edit, not a format for submission. The `fonts_embedded` column is the one to keep in a submission script: `cairo_pdf` embeds the glyphs, the default `pdf()` device references them by name, and that difference is the difference between a figure that renders identically everywhere and one that silently substitutes a font on the publisher's system.

<div class="exercise"><h4>Exercise 1</h4>
<p>Re-run the chunk above after replacing <code>geom_point()</code> with a rasterised layer (draw the points into a PNG with <code>ragg::agg_png</code> at 600 dpi and compose it with the vector axes, or simply export the whole figure as PNG) and compare the PDF size. Then increase the number of points by plotting the full <code>df</code>, censored compounds included, and compare again. At what point does rasterising the data layer become the right decision, and what do you lose when you make it?</p></div>

## 3. The same axes at two sizes: why the declared size is not a detail

The clearest way to internalise the sizing rule is to see it fail. The next figure draws identical content twice with identical font *settings*. The left panel is what the reader sees when the figure was drawn at one-column width; the right panel is the same figure drawn at full-page width and then reduced by the typesetter to one column, which is exactly what happens when you hand in a plot made at the device default.



In [ ]:
shrink <- 8.5 / 17.4
base_plot <- function(bs, lab) {
  ggplot(ex, aes(MolWt, pActivity)) +
    geom_point(size = 0.35, alpha = 0.2, colour = PAL[["blue"]]) +
    labs(x = "Molecular weight (Da)", y = "pActivity", title = lab) +
    theme_minimal(base_size = bs) +
    theme(plot.title = element_text(face = "bold", size = bs * 1.1), panel.grid.minor = element_blank())
}
pA <- base_plot(9, sprintf("drawn at 8.5 cm: labels arrive as %.1f pt", 9))
pB <- base_plot(9 * shrink, sprintf("drawn at 17.4 cm, printed at 8.5 cm: %.1f pt", 9 * shrink))
print((pA | pB) + plot_annotation(title = "Identical font settings, two final widths",
                                  theme = theme(plot.title = element_text(face = "bold", size = 10))))
cat(sprintf("scale factor %.2f; a 9 pt label becomes %.1f pt, a 7 pt tick label becomes %.1f pt\n",
            shrink, 9 * shrink, 7 * shrink))



<p class="caption"><b>Figure 1.</b> The same scatter of molecular weight against pActivity rendered for two final widths. Both panels occupy the same area on this page. The right-hand panel was drawn at full-page width and reduced by the factor 0.49 that a typesetter applies when the figure is placed in one column; its 9-point labels arrive at about 4.4 points and its 7-point tick labels at 3.4 points, below every publisher's minimum. The content is identical: only the declared width differs. Draw at the final width, choose the font size once, and never scale the figure again.</p>

## 4. A theme, not a hundred arguments

Consistency across the figures of one paper is itself a quality signal, and it is impossible to achieve by passing arguments at each call site. In `ggplot2`<sup class="c"><a href="#r3">[3]</a></sup> the mechanism is a **theme**: a function returning a set of defaults that can be installed globally with `theme_set()`, written to an R file and shipped with the manuscript's repository — the equivalent of a matplotlib stylesheet in the Python edition of this unit. The laboratory theme below is deliberately austere: no panel border, a small sans-serif font, a light grid only on the value axis, and a figure size expressed in centimetres wherever the figure is saved. Colour is handled separately, by setting the Okabe–Ito palette<sup class="c"><a href="#r8">[8]</a></sup> as the default discrete scale.



In [ ]:
theme_lbqc <- function(base_size = 8) {
  theme_minimal(base_size = base_size) +
    theme(plot.title    = element_text(face = "bold", size = base_size, hjust = 0),
          axis.title    = element_text(size = base_size),
          axis.text     = element_text(size = base_size - 1),
          legend.text   = element_text(size = base_size - 1),
          legend.title  = element_text(size = base_size - 1),
          legend.key.size = unit(0.35, "cm"),
          panel.grid.minor = element_blank(),
          panel.grid.major = element_line(linewidth = 0.25, colour = "grey85"),
          axis.line     = element_line(linewidth = 0.3, colour = "grey20"),
          plot.margin   = margin(2, 2, 2, 2))
}
style_file <- file.path(WORK, "lbqc_theme.R")
writeLines(c(deparse(theme_lbqc), "",
             'options(ggplot2.discrete.colour = OKABE_ITO, ggplot2.discrete.fill = OKABE_ITO)'),
           con = style_file)
theme_set(theme_lbqc())
options(ggplot2.discrete.colour = OKABE_ITO, ggplot2.discrete.fill = OKABE_ITO)
n_set <- sum(!vapply(theme_lbqc(), is.null, logical(1)))
cat(sprintf("the laboratory theme fixes %d of the %d theme elements ggplot2 exposes\n",
            n_set, length(theme_grey())))
cat(sprintf("written to %s (%d lines, %.1f kB); reload it in any script with source()\n",
            basename(style_file), length(readLines(style_file)), file.size(style_file) / 1024))
cat("first three default colours:", paste(OKABE_ITO[1:3], collapse = ", "), "\n")



Two details are worth defending. The theme is a *function* with a `base_size` argument, not a fixed object, so a figure that will be printed across two columns can be drawn with `theme_lbqc(9)` while a four-panel figure uses `theme_lbqc(7)` — the sizes stay explicit and the rest of the appearance stays identical. And the palette is installed through `options()` rather than added to each plot, so a figure that forgets `scale_colour_manual()` still comes out in colour-vision-safe colours. Writing the theme to a file with `deparse()` and reloading it with `source()` is the R equivalent of shipping a stylesheet: the manuscript repository then contains the appearance of its own figures.

## 5. Remaking an exploratory figure as a publication figure

Here is the worked case. Unit 5 produced an exploratory figure while looking for structure in the ROCK1 table: molecular weight against pActivity, coloured by a third variable, with default styling and no thought given to overplotting. It did its job — it was looked at once, by one person, for ten seconds. Printed in a paper it would be unreadable and, worse, misleading: with thousands of semi-transparent points the eye sees the outline of the cloud rather than its density, and a rainbow colour map makes a continuous variable look like it has bands.

The remake applies six changes, each of which can be justified to a referee: final size and font set by the laboratory theme; overplotting solved by binning rather than transparency; the rainbow map replaced by a perceptually uniform one<sup class="c"><a href="#r6">[6]</a><a href="#r7">[7]</a></sup>; a second panel that shows the quantity the paper actually claims (predicted against observed under the scaffold split) instead of a descriptive correlation; reference lines that encode the claim (the identity line and the experimental error band); and panel letters so the caption can refer to them.



In [ ]:
print(ggplot(ex, aes(MolWt, pActivity, colour = MolLogP)) +
  geom_point(alpha = 0.6, size = 1.8) +
  scale_colour_gradientn(colours = rainbow(7)) +        ## the rainbow map, kept here to be criticised
  labs(x = "MolWt", y = "pActivity", title = "MolWt vs pActivity colored by MolLogP") +
  theme_grey(base_size = 11))
cat(sprintf("%d points drawn on %.0f square inches; Pearson r(MolWt, pActivity) = %+.3f\n",
            nrow(ex), 6.4 * 4.2, cor(ex$MolWt, ex$pActivity)))



<p class="caption"><b>Figure 2.</b> The exploratory version. Three defects matter for print: the figure is drawn at the device default of about 16 cm and would be reduced by half, the rainbow colour map invents visual boundaries in a smooth variable and is unreadable in greyscale, and the thousands of overlapping markers hide the density where the data are densest. The correlation printed beneath it is weak, which is itself the finding this panel should communicate — and does not.</p>

Now the publication version of the same evidence. The hexagonal binning of the Python edition needs the `hexbin` package, which is not installed in this environment, so the density panel uses `geom_bin2d()` — square bins of the same idea, with the count encoded by a perceptually uniform fill.



In [ ]:
med <- ex[, .(med = median(pActivity)), by = .(bin = cut(MolWt, seq(150, 750, by = 50)))][!is.na(bin)]
med[, centre := seq(175, 725, by = 50)[as.integer(bin)]]
pA <- ggplot(ex, aes(MolWt, pActivity)) +
  geom_bin2d(bins = 34) +
  scale_fill_viridis_c(name = "compounds\nper bin") +
  geom_line(data = med, aes(centre, med), colour = "#D55E00", linewidth = 0.6) +
  geom_point(data = med, aes(centre, med), colour = "#D55E00", size = 0.9) +
  coord_cartesian(xlim = c(150, 750)) +
  labs(x = "Molecular weight (Da)", y = expression(pActivity~(-log[10]~M)))
lims <- c(3.3, 10.2)
pB <- ggplot(data.frame(obs = yt, pred = pred_te), aes(obs, pred)) +
  geom_ribbon(data = data.frame(obs = lims), inherit.aes = FALSE,
              aes(x = obs, ymin = obs - sigma_exp, ymax = obs + sigma_exp), fill = "grey50", alpha = 0.18) +
  geom_point(size = 0.5, alpha = 0.5, colour = "#0072B2") +
  geom_abline(slope = 1, intercept = 0, linewidth = 0.35) +
  annotate("text", x = 10.0, y = 3.6, hjust = 1, vjust = 0, size = 2.4,
           label = sprintf("RMSE %.2f\nR2 %.2f\nn = %d", rmse_sc, r2_sc, length(TE))) +
  coord_equal(xlim = lims, ylim = lims) +
  labs(x = "Observed pActivity", y = "Predicted pActivity",
       subtitle = sprintf("grey band: +/- sigma_exp = %.2f", sigma_exp)) +
  theme(plot.subtitle = element_text(size = 6))
fig1 <- (pA | pB) + plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(face = "bold", size = 10))
print(fig1)
ggsave(file.path(WORK, "figure1.pdf"), fig1, width = 17.4, height = 7.0, units = "cm", device = cairo_pdf)
cat(sprintf("saved figure1.pdf, %.0f kB, drawn at 17.4 x 7.0 cm; densest bin holds %d compounds\n",
            file.size(file.path(WORK, "figure1.pdf")) / 1024,
            max(ggplot_build(pA)$data[[1]]$count)))



<p class="caption"><b>Figure 3.</b> The publication remake. <b>(A)</b> Density of the 2,764 exact-value ROCK1 compounds in the molecular-weight / potency plane, binned into squares so that the colour encodes how many compounds occupy each region; the orange line is the median potency per 50 Da bin and is nearly flat across the range, which is the honest reading of a correlation of about +0.08. <b>(B)</b> Predicted against observed pActivity for the scaffold hold-out set, with the identity line and a grey band of one experimental standard deviation; the model's scaffold-split RMSE is printed in the panel. Both panels were produced by the code shown, exported at 17.4 × 7.0 cm with embedded fonts, with sizes and colours fixed by the laboratory theme.</p>

Compare the two captions. The exploratory caption describes what is plotted; the publication caption describes what is plotted, states the sample size, names the statistical object (median per bin, identity line, error band) and tells the reader what to conclude. That is the **caption contract**: a caption must let a reader who skipped the text understand the panel, and it must not assert anything the panel does not show.

## 6. Colour, accessibility and the greyscale test

Roughly one man in twelve and one woman in two hundred has a colour-vision deficiency, most often a reduced ability to distinguish red from green. A figure that encodes its only distinction in red versus green is unreadable to them, and a figure printed in greyscale — which still happens, and always happens when a reader photocopies a page — is unreadable to everyone if its categories differ only in hue. Two rules remove the whole problem: use a perceptually uniform sequential map (viridis, cividis, magma) for continuous variables, and a categorical palette designed for colour-vision deficiency, such as the Okabe–Ito set installed in Section 4<sup class="c"><a href="#r8">[8]</a></sup>, for discrete ones — plus a second channel (point shape, line type) whenever the distinction carries the claim<sup class="c"><a href="#r6">[6]</a></sup>.

The chunk below simulates deuteranopia with a standard linear transformation of the sRGB values and converts the same colours to luminance, so that the failure can be seen rather than taken on faith.



In [ ]:
DEUTER <- matrix(c(0.625, 0.375, 0.0, 0.70, 0.30, 0.0, 0.0, 0.30, 0.70), nrow = 3, byrow = TRUE)
simulate_cvd <- function(cols, kind) {
  rgbm <- t(col2rgb(cols) / 255)
  out <- if (kind == "deuteranopia") pmin(pmax(rgbm %*% t(DEUTER), 0), 1) else
    matrix(rep(as.vector(rgbm %*% c(0.2126, 0.7152, 0.0722)), 3), ncol = 3)
  rgb(out[, 1], out[, 2], out[, 3])
}
BAD <- c("#B22222", "#556B2F", "#CD5C5C", "#6B8E23", "#A0522D", "#2E6B2E")
pals <- list("red/green palette (bad)" = BAD, "Okabe-Ito palette (good)" = OKABE_ITO[1:6],
             "rainbow colour map (bad)" = rainbow(6), "viridis colour map (good)" = hcl.colors(6, "viridis"))
grid_df <- do.call(rbind, lapply(names(pals), function(nm) do.call(rbind, lapply(
  c("original", "deuteranopia", "greyscale"), function(kd) data.frame(
    palette = nm, condition = kd, i = 1:6,
    col = if (kd == "original") pals[[nm]] else simulate_cvd(pals[[nm]], kd))))))
grid_df$palette   <- factor(grid_df$palette, levels = names(pals))
grid_df$condition <- factor(grid_df$condition, levels = c("original", "deuteranopia", "greyscale"))
print(ggplot(grid_df, aes(i, 1, fill = col)) + geom_tile() + scale_fill_identity() +
  facet_grid(palette ~ condition, switch = "y") +
  theme_void(base_size = 8) +
  theme(strip.text.y.left = element_text(angle = 0, hjust = 1, size = 6.5),
        strip.text.x = element_text(size = 8)))
d1 <- col2rgb(simulate_cvd(BAD[1:2], "deuteranopia")) / 255
d2 <- col2rgb(simulate_cvd(OKABE_ITO[1:2], "deuteranopia")) / 255
gj <- col2rgb(simulate_cvd(rainbow(6)[c(1, 6)], "greyscale")) / 255    ## the two ends of the map
gv <- col2rgb(simulate_cvd(hcl.colors(6, "viridis")[c(1, 6)], "greyscale")) / 255
cat(sprintf("firebrick vs olive under deuteranopia: distance %.2f\n", sqrt(sum((d1[, 1] - d1[, 2])^2))))
cat(sprintf("Okabe-Ito blue vs vermillion: distance %.2f (%.1f times larger)\n",
            sqrt(sum((d2[, 1] - d2[, 2])^2)), sqrt(sum((d2[, 1] - d2[, 2])^2)) / sqrt(sum((d1[, 1] - d1[, 2])^2))))
cat(sprintf("rainbow, first and last colour: greyscale luminance %.3f vs %.3f\n", gj[1, 1], gj[1, 2]))
cat(sprintf("viridis, first and last colour: greyscale luminance %.3f vs %.3f\n", gv[1, 1], gv[1, 2]))



<p class="caption"><b>Figure 4.</b> Four palettes under three conditions. Rows one and three use colours that collapse: firebrick and olive, chosen with similar lightness, become nearly the same colour for a deuteranope, and the rainbow map is non-monotonic in luminance, so its first and last colours — the two ends of the scale — print as nearly the same grey. Rows two and four stay separable in all three columns. The printed numbers quantify it — the Okabe–Ito pair keeps several times the separation of the red/green pair under the same simulation, and viridis, whose luminance rises monotonically, separates its two ends in greyscale by an order of magnitude more.</p>

## 7. Figures that mislead, and the honest version of the same data

The most damaging figure is not the ugly one; it is the one that is beautiful and overstates the evidence. Three patterns account for most of it in computational papers: a truncated value axis that turns a small difference into a large one, a bar of means that hides the distribution it averages<sup class="c"><a href="#r5">[5]</a></sup>, and a dual-axis plot that invites the reader to see a correlation between two arbitrary scalings. The next chunk builds the first two from real data — two scaffold families from the ROCK1 table — and puts the honest version beside each.



In [ ]:
cnt <- sort(table(scaf), decreasing = TRUE); fam <- names(cnt[cnt >= 30])
a <- ex[scaffold_smiles == fam[3], pActivity]; b <- ex[scaffold_smiles == fam[4], pActivity]
dmean <- mean(a) - mean(b)
pooled_sd <- sqrt(((length(a) - 1) * var(a) + (length(b) - 1) * var(b)) / (length(a) + length(b) - 2))
cohen_d <- dmean / pooled_sd
set.seed(1); boot <- replicate(2000, mean(sample(a, length(a), TRUE)) - mean(sample(b, length(b), TRUE)))
ci <- quantile(boot, c(0.025, 0.975))
dd <- data.frame(fam = rep(c("scaffold A", "scaffold B"), c(length(a), length(b))), v = c(a, b))
ms <- data.frame(fam = c("scaffold A", "scaffold B"), m = c(mean(a), mean(b)),
                 se = c(sd(a) / sqrt(length(a)), sd(b) / sqrt(length(b))))
p1 <- ggplot(ms, aes(fam, m, fill = fam)) + geom_col(width = 0.5, show.legend = FALSE) +
  coord_cartesian(ylim = c(min(ms$m) - 0.05, max(ms$m) + 0.05)) +
  labs(x = NULL, y = "mean pActivity", title = "truncated axis (misleading)")
p2 <- ggplot(ms, aes(fam, m, fill = fam)) + geom_col(width = 0.5, show.legend = FALSE) +
  geom_errorbar(aes(ymin = m - se, ymax = m + se), width = 0.15) + ylim(0, 10) +
  labs(x = NULL, y = "mean pActivity", title = "bar of means (hides spread)")
set.seed(2)
p3 <- ggplot(dd, aes(fam, v, fill = fam)) +
  geom_violin(alpha = 0.35, colour = NA, show.legend = FALSE) +
  geom_jitter(width = 0.08, size = 0.5, alpha = 0.5, show.legend = FALSE) +
  scale_x_discrete(labels = c(sprintf("scaffold A\n(n=%d)", length(a)), sprintf("scaffold B\n(n=%d)", length(b)))) +
  labs(x = NULL, y = "pActivity", title = "distribution (honest)")
print(p1 | p2 | p3)
cat(sprintf("difference of means %+.3f log units, 95%% bootstrap CI [%+.2f, %+.2f], Cohen's d = %+.2f\n",
            dmean, ci[1], ci[2], cohen_d))
cat(sprintf("apparent bar-height ratio with the truncated axis: %.1fx; true ratio of the means: %.2fx\n",
            (mean(a) - (min(ms$m) - 0.05)) / (mean(b) - (min(ms$m) - 0.05)), mean(a) / mean(b)))
cat(sprintf("experimental noise for a single compound is sigma_exp = %.2f log units, %.1f times the difference plotted\n",
            sigma_exp, abs(sigma_exp / dmean)))



<p class="caption"><b>Figure 5.</b> The same comparison between two Bemis–Murcko scaffold families of comparable size (43 and 38 compounds), drawn three ways. The left panel truncates the value axis and makes the left bar look several times the right one; the middle panel uses a full axis but replaces each distribution with a mean and a standard error, which hides that the two groups overlap almost completely; the right panel shows every compound. The printed statistics settle it: the difference of means is a fraction of a log unit, its 95 % bootstrap confidence interval includes zero, Cohen's <i>d</i> is small, and the experimental noise of a single measurement is several times the difference being plotted.</p>

This is the unit's failure-mode experiment, and it is worth being explicit about what failed. Nothing in the left panel is false: the bars are at the right heights, the axis is labelled, the data are real. The panel is dishonest because of what it omits — the zero, the spread and the sample size — and because the reader's visual system reads bar *length* as magnitude<sup class="c"><a href="#r2">[2]</a></sup>. The rule the laboratory applies is mechanical: bars start at zero or are not bars; a mean is never shown without its distribution or at least its confidence interval; and a difference is never plotted without the noise floor it must beat.

<div class="exercise"><h4>Exercise 2</h4>
<p>Build the third misleading pattern yourself: plot the mean molecular weight and the mean pActivity of the compounds in each decile of <code>n_measurements</code> on two axes with <code>sec_axis()</code>, and choose the two scalings so the curves appear to track each other. Then show the same data as two small multiples sharing an x axis (<code>facet_wrap(~ variable, scales = "free_y", ncol = 1)</code>), and write the one-sentence caption that each version implies. Which of the two would you defend to a referee?</p></div>

## 8. Multi-panel figures and the order in which they are read

A paper in our fields usually has four to six figures, and each one answers one question. Within a figure, panels are read left to right and top to bottom, and the reader assumes that this order is an argument: panel A establishes the setting, B shows the main evidence, C the control, D the consequence. Laying out panels that do not form such a sequence wastes the strongest convention you have.

Mechanically, multi-panel layouts in R are solved by `patchwork`, which composes complete `ggplot2` objects — themselves built on the grammar of graphics<sup class="c"><a href="#r4">[4]</a></sup>, so a panel is a composition of data, scales and marks rather than a picture — with the operators `|` (side by side), `/` (stacked) and `+`, and adds panel letters in one call with `plot_annotation(tag_levels = "A")`. That single call is what replaces the hand-placed text labels of a drawing program, and it is why renumbering a panel costs nothing. The figure below is the Figure 1 of a hypothetical ROCK1 paper, assembled from objects this document has already computed.



In [ ]:
resid <- pred_te - yt
qA <- ggplot(ex, aes(pActivity)) +
  geom_histogram(bins = 40, fill = "#0072B2", alpha = 0.85) +
  geom_vline(xintercept = median(ex$pActivity), colour = "#D55E00", linewidth = 0.5) +
  labs(x = expression(pActivity~(-log[10]~M)), y = "compounds", title = "Potency distribution")
qB <- ggplot(data.frame(obs = yt, pred = pred_te), aes(obs, pred)) +
  geom_point(size = 0.4, alpha = 0.45, colour = "#0072B2") +
  geom_abline(slope = 1, intercept = 0, linewidth = 0.3) + coord_equal(xlim = lims, ylim = lims) +
  labs(x = "Observed", y = "Predicted", title = "Scaffold hold-out")
qC <- ggplot(data.frame(r = resid), aes(r)) +
  geom_histogram(bins = 35, fill = "#009E73", alpha = 0.85) +
  geom_vline(xintercept = 0, linewidth = 0.3) +
  geom_vline(xintercept = c(-sigma_exp, sigma_exp), colour = "#D55E00", linetype = 2, linewidth = 0.4) +
  labs(x = "Residual (log units)", y = "compounds", title = "Residuals vs experimental noise")
ks <- seq(10, 200, by = 10); ordp <- order(-pred_te)
hits <- vapply(ks, function(kk) mean(yt[ordp[1:kk]] >= 7), numeric(1)); base_rate <- mean(yt >= 7)
qD <- ggplot(data.frame(k = ks, p = hits), aes(k, p)) +
  geom_line(colour = "#0072B2", linewidth = 0.5) + geom_point(colour = "#0072B2", size = 0.8) +
  geom_hline(yintercept = base_rate, colour = "grey40", linetype = 3) +
  annotate("text", x = 200, y = base_rate + 0.04, hjust = 1, size = 2.2,
           label = sprintf("random baseline (%.2f)", base_rate)) +
  ylim(0, 1) + labs(x = "Top-k compounds selected", y = expression(Fraction~with~pActivity>=7),
                    title = "Enrichment of a selection")
fig2 <- (qA | qB) / (qC | qD) + plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(face = "bold", size = 10))
print(fig2)
ggsave(file.path(WORK, "figure2_fourpanel.pdf"), fig2, width = 17.4, height = 12.0,
       units = "cm", device = cairo_pdf)
cat(sprintf("panel D: top-10 precision %.2f vs baseline %.2f (enrichment %.1fx); top-200 precision %.2f\n",
            hits[1], base_rate, hits[1] / base_rate, hits[length(hits)]))
cat(sprintf("panel C: %.0f%% of residuals fall within one sigma_exp\n", 100 * mean(abs(resid) <= sigma_exp)))



<p class="caption"><b>Figure 6.</b> A complete four-panel figure assembled from the analysis with <code>patchwork</code>. <b>(A)</b> Distribution of measured potency over the 2,764 exact-value compounds, median marked. <b>(B)</b> Predicted against observed potency on the scaffold hold-out, identity line shown. <b>(C)</b> Residual distribution with dashed lines at plus and minus one experimental standard deviation; the fraction of residuals inside that band is printed below the figure. <b>(D)</b> Precision of the top-<i>k</i> ranked compounds against the base rate, the quantity that decides whether the model is useful for selecting compounds. The panels form an argument: this is the data, this is how well the model does on new scaffolds, this is how its error compares with the measurement itself, and this is what that buys when choosing compounds to test.</p>

The convention extends to the figures themselves: Figure 1, the data; Figure 2, the method or its validation; Figure 3, the main result; Figure 4, the control that excludes the obvious alternative; Figure 5, the application. A figure that cannot be placed in that sequence usually belongs in the supplement. The diagram below sketches the same logic at the level of sentences.



In [ ]:
boxes <- data.frame(
  x = c(0.18, 0.50, 0.82), lab = c("Question", "Evidence", "Claim"),
  txt = c("Does a fingerprint model\npredict ROCK1 potency for\nchemotypes it has not seen?",
          "Scaffold hold-out:\nRMSE, residuals vs noise,\ntop-k precision",
          "Useful for prioritising\nnew chemotypes, within\nthe stated domain"))
arrows_df <- data.frame(x = c(0.315, 0.635), xend = c(0.365, 0.685), y = 0.55)
notes <- data.frame(x = c(0.34, 0.66), y = 0.18,
                    txt = c("broken by:\nthe wrong split,\na leaked feature",
                            "broken by:\nover-claiming,\nmetric cherry-picking"))
print(ggplot() +
  geom_rect(data = boxes, aes(xmin = x - 0.13, xmax = x + 0.13, ymin = 0.33, ymax = 0.80),
            fill = "#eef3fa", colour = "#0072B2", linewidth = 0.5) +
  geom_text(data = boxes, aes(x, 0.74, label = lab), fontface = "bold", size = 3.4) +
  geom_text(data = boxes, aes(x, 0.54, label = txt), size = 2.5, lineheight = 1.05) +
  geom_segment(data = arrows_df, aes(x = x, xend = xend, y = y, yend = y),
               arrow = arrow(length = unit(0.18, "cm"), type = "closed"), linewidth = 0.5) +
  geom_text(data = notes, aes(x, y, label = txt), colour = "#b32d00", size = 2.4, lineheight = 1.05) +
  annotate("text", x = 0.5, y = 0.93, size = 3, fontface = "italic",
           label = "Every sentence of a Results section sits on one of these three steps") +
  coord_cartesian(xlim = c(0.02, 0.98), ylim = c(0.08, 1)) + theme_void())
cat("the two red annotations are the two failures a referee is specifically looking for\n")



<p class="caption"><b>Figure 7.</b> The argument skeleton of a computational paper. The link from question to evidence is broken by a validation design that does not match the question — a random split used to support a claim about new chemotypes, or a feature computed before the split. The link from evidence to claim is broken by language: "accurately predicts" where the evidence supports "ranks better than chance within the applicability domain".</p>

## 9. Tables, significant figures and the uncertainty that sets them

Tables are read more carefully than figures and written more carelessly. Three rules cover almost everything. **Units go in the header, never in the cells.** **Columns are aligned on the decimal point**, with the same number of decimals throughout a column. And **significant figures are set by the uncertainty, not by the width of the double**: printing `RMSE = 0.85753` claims one part in a hundred thousand for a quantity whose confidence interval is tenths wide.

The rule that sets the significant figures needs an uncertainty, so a performance table needs confidence intervals before it needs formatting. For a hold-out metric the cheap and honest answer is a bootstrap over the test compounds: resample with replacement, recompute the metric, take the 2.5th and 97.5th percentiles. It measures the uncertainty due to *which compounds happened to be in the test set* — not the uncertainty due to the choice of split, which is larger and is addressed by repeating the split. The same resampling gives an uncertainty for the cross-validated \( R^{2} \), and the top-10 precision of Section 7 has the binomial standard error of a proportion over ten compounds.



In [ ]:
set.seed(0)
BOOT <- 2000; n_te <- length(yt)
idx  <- matrix(sample.int(n_te, BOOT * n_te, replace = TRUE), nrow = BOOT)
boot_ci <- function(fn) { v <- apply(idx, 1, fn); c(mean(v), quantile(v, c(0.025, 0.975))) }
ci_rmse <- boot_ci(function(i) sqrt(mean((pred_te[i] - yt[i])^2)))
ci_mae  <- boot_ci(function(i) mean(abs(pred_te[i] - yt[i])))
ci_r2   <- boot_ci(function(i) 1 - sum((pred_te[i] - yt[i])^2) / sum((yt[i] - mean(yt[i]))^2))
ci_rho  <- boot_ci(function(i) suppressWarnings(cor(pred_te[i], yt[i], method = "spearman")))
rho_sc  <- cor(pred_te, yt, method = "spearman")
half    <- function(ci) (ci[3] - ci[2]) / 2
r2cv_boot <- replicate(1000, { i <- sample.int(length(y), length(y), TRUE)
                               1 - sum((pred_cv[i] - y[i])^2) / sum((y[i] - mean(y[i]))^2) })
unc_r2cv  <- sd(r2cv_boot)                        ## tight enough that its leading digit is a 1
unc_p10   <- sqrt(hits[1] * (1 - hits[1]) / 10)   ## binomial standard error of 10 selected compounds
cat(sprintf("scaffold RMSE     %.5f +/- %.5f  (half-width of the 95%% bootstrap interval)\n",
            rmse_sc, half(ci_rmse)))
cat(sprintf("random-CV R2      %.5f +/- %.5f  (bootstrap standard deviation over all compounds)\n",
            r2_cv, unc_r2cv))
cat(sprintf("top-10 precision  %.5f +/- %.5f  (binomial standard error)\n", hits[1], unc_p10))



Those three pairs are the worked example. The convention used in the physical sciences is: round the uncertainty to one significant figure (two if its leading digit is 1), then round the value to the same decimal place. Work it by hand before writing the function.

| quantity | raw value | raw uncertainty | step 1: round the uncertainty | step 2: match the value | reported |
|---|---|---|---|---|---|
| scaffold RMSE | 0.85753… | 0.05180… | 0.05 (1 s.f., leading digit 5) | 2 decimals | 0.86 ± 0.05 |
| random-CV \( R^{2} \) | 0.57132… | 0.01147… | 0.011 (2 s.f., leading digit 1) | 3 decimals | 0.571 ± 0.011 |
| top-10 precision | 0.80000 | 0.12649… | 0.13 (2 s.f., leading digit 1) | 2 decimals | 0.80 ± 0.13 |

The second and third rows are the ones students get wrong: when the uncertainty begins with a 1, one significant figure would discard almost half the information in it, so two are kept. The next chunk implements the rule and checks it against the table above, with the live quantities rather than copies of them.



In [ ]:
round_to_uncertainty <- function(value, unc) {
  if (!is.finite(unc) || unc <= 0) return(c(format(value, digits = 3), ""))
  e    <- floor(log10(abs(unc)))
  lead <- as.integer(abs(unc) / 10^e)
  sig  <- if (lead == 1) 2 else 1
  dec  <- max(0, -(e - (sig - 1)))
  c(sprintf("%.*f", dec, value), sprintf("%.*f", dec, round(unc, dec)))
}
checks <- list(list("scaffold RMSE", rmse_sc, half(ci_rmse), "0.86", "0.05"),
               list("random-CV R2", r2_cv, unc_r2cv, "0.571", "0.011"),
               list("top-10 precision", hits[1], unc_p10, "0.80", "0.13"))
for (ch in checks) {
  got <- round_to_uncertainty(ch[[2]], ch[[3]])
  cat(sprintf("%-18s %.5f +/- %.5f  ->  %s +/- %s   %s\n", ch[[1]], ch[[2]], ch[[3]],
              got[1], got[2],
              if (identical(got, c(ch[[4]], ch[[5]]))) "matches the hand-worked table" else "MISMATCH"))
}



All three rows reproduce the hand-worked table, including the two cases where the uncertainty begins with a 1 and keeps a second significant figure. The table itself is then assembled from the same objects, so no number in it was typed.



In [ ]:
mk <- function(lab, val, ci) {
  g <- round_to_uncertainty(val, half(ci))
  data.frame(Metric = lab, Value = g[1], `95% CI` = sprintf("[%.2f, %.2f]", ci[2], ci[3]),
             `+/-` = g[2], check.names = FALSE)
}
perf <- rbind(mk("RMSE (log units)", rmse_sc, ci_rmse), mk("MAE (log units)", mae_sc, ci_mae),
              mk("R2", r2_sc, ci_r2), mk("Spearman rho", rho_sc, ci_rho))
kable(perf, align = "lrrr")


<p class="caption"><b>Table 1.</b> Performance of the random-forest model on the scaffold hold-out set (553 compounds), with 95 % bootstrap confidence intervals over 2,000 resamples of the test set. Values are rounded to the decimal implied by their own uncertainty.</p>



In [ ]:
cat(sprintf("MAE %.3f log units vs experimental floor sigma_exp %.3f: the model's MAE is %.2f times it\n",
            mae_sc, sigma_exp, mae_sc / sigma_exp))
cat(sprintf("RMSE %.3f [%.3f, %.3f]; R2 %.3f [%.3f, %.3f]; Spearman rho %.3f [%.3f, %.3f]\n",
            rmse_sc, ci_rmse[2], ci_rmse[3], r2_sc, ci_r2[2], ci_r2[3], rho_sc, ci_rho[2], ci_rho[3]))



The MAE is close to the pooled experimental standard deviation of the data, which is the correct reference point for judging whether the model is worth deploying: it predicts an unseen scaffold about as well as one laboratory reproduces another. A performance table is also the place where **metric cherry-picking** happens. Reporting \( R^{2} \) alone, on the split that flatters the model, is the most common form; reporting only \( R^{2} \) at all is nearly as bad, because \( R^{2} \) depends on the variance of the test set and so changes when the split changes even if the errors do not. Report an error in the units of the target (RMSE and MAE in log units), one rank metric, the sample size and the split. The figure below shows why: the same model, three evaluation protocols, three different stories.



In [ ]:
set.seed(0)
cv_boot <- replicate(1000, { i <- sample.int(length(y), length(y), TRUE); sqrt(mean((pred_cv[i] - y[i])^2)) })
ci_cv   <- quantile(cv_boot, c(0.025, 0.975))
dummy   <- sqrt(mean((y - mean(y))^2))
fp <- data.frame(label = factor(c("mean-prediction baseline", "random 5-fold CV", "scaffold hold-out"),
                                levels = c("scaffold hold-out", "random 5-fold CV", "mean-prediction baseline")),
                 v = c(dummy, rmse_cv, rmse_sc),
                 lo = c(dummy, ci_cv[1], ci_rmse[2]), hi = c(dummy, ci_cv[2], ci_rmse[3]))
print(ggplot(fp, aes(v, label)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.12, colour = "#0072B2", linewidth = 0.7) +
  geom_point(colour = "#0072B2", size = 2) +
  geom_text(aes(label = sprintf("%.3f", v)), vjust = -1.1, size = 2.6) +
  geom_vline(xintercept = sigma_exp, colour = "#D55E00", linetype = 2, linewidth = 0.5) +
  annotate("text", x = sigma_exp + 0.01, y = 0.62, hjust = 0, colour = "#D55E00", size = 2.4,
           label = sprintf("experimental floor %.2f", sigma_exp)) +
  xlim(0, 1.25) + labs(x = "RMSE (log units)", y = NULL, title = "The same model under three protocols"))
cat(sprintf("baseline %.3f | random CV %.3f [%.3f, %.3f] | scaffold %.3f [%.3f, %.3f]\n",
            dummy, rmse_cv, ci_cv[1], ci_cv[2], rmse_sc, ci_rmse[2], ci_rmse[3]))
cat(sprintf("choosing the random-CV number over the scaffold number improves the headline by %.0f%% without changing the model\n",
            100 * (rmse_sc - rmse_cv) / rmse_sc))



<p class="caption"><b>Figure 8.</b> Root-mean-square error of one random forest under three protocols, with 95 % bootstrap intervals and the experimental noise floor as a dashed line. The baseline is the error of predicting the training mean for everything. The gap between the random cross-validation and the scaffold hold-out is a property of the <i>evaluation</i>, not of the model; a paper that quotes only the middle point improves its headline number by about a sixth while telling the reader nothing about new chemotypes.</p>

## 10. From a data frame to LaTeX, Word and a machine-readable supplement

Three destinations, three formats, one source of truth. The data frame is the source; everything else is generated. `knitr::kable(format = "latex", booktabs = TRUE)` produces a table that compiles unmodified in a journal template; `format = "pipe"` produces a Markdown table that can be pasted into a Word manuscript or a GitHub README; and the supplement is a CSV — not a spreadsheet — with a companion data dictionary, because a `.xlsx` file converts gene symbols into dates and compound identifiers into floats<sup class="c"><a href="#r28">[28]</a></sup>.



In [ ]:
cat(kable(perf, format = "latex", booktabs = TRUE, align = "lrrr",
          label = "performance",
          caption = "Performance of the ROCK1 random-forest model on the scaffold hold-out set (n = 553), with 95% bootstrap confidence intervals."),
    sep = "\n")
cat("\n\n")
cat(kable(perf, format = "pipe", align = "lrrr"), sep = "\n")



The LaTeX block above is ready for a journal template that loads `booktabs`; the Markdown block below it is what goes into a README or a Word document through a paste. Neither was typed by hand, so neither can disagree with Table 1.



In [ ]:
supp <- data.frame(inchikey = ex$inchikey[TE], std_smiles = ex$std_smiles[TE],
                   scaffold_smiles = ex$scaffold_smiles[TE],
                   pActivity_observed = round(yt, 3), pActivity_predicted = round(pred_te, 3),
                   residual = round(pred_te - yt, 3), n_measurements = ex$n_measurements[TE])
fwrite(supp, file.path(WORK, "supplementary_table_S1.csv"))
dd <- data.frame(
  column = names(supp),
  type = c("string", "string", "string", "float", "float", "float", "integer"),
  units = c("", "", "", "-log10(M)", "-log10(M)", "-log10(M)", "count"),
  description = c("InChIKey of the standardised parent structure (row key)",
                  "Canonical SMILES after RDKit normalisation, largest fragment, uncharging",
                  "Bemis-Murcko generic scaffold; defines the hold-out split",
                  "Median of the exact measurements in ChEMBL 37",
                  "Random-forest prediction, model version 1.0.0",
                  "predicted minus observed", "Number of source measurements aggregated into the row"))
fwrite(dd, file.path(WORK, "supplementary_table_S1_dictionary.csv"))
sha_s1 <- digest(file.path(WORK, "supplementary_table_S1.csv"), algo = "sha256", file = TRUE)
kable(head(supp, 4), caption = "First rows of Supplementary Table S1")
kable(dd, caption = "Data dictionary of Supplementary Table S1")
cat(sprintf("S1: %d rows, %.0f kB, SHA-256 %s...\n", nrow(supp),
            file.size(file.path(WORK, "supplementary_table_S1.csv")) / 1024, substr(sha_s1, 1, 16)))



Three things make that supplement reusable rather than decorative: a stable identifier (the InChIKey) to join on, the structure itself so the join works for a reader who does not use InChIKeys, and a data dictionary naming the type, unit and provenance of every column. The checksum goes in the repository manifest, so a file downloaded two years later can be shown to be the file that was published.

## 11. A Methods section that permits replication

The test of a Methods section is simple and brutal: could a competent stranger, with your data and no access to you, obtain your numbers? For a computational paper that requires five classes of information — the data and its provenance, the preprocessing, the model and every hyperparameter that is not a default, the validation protocol, and the software environment with versions and seeds<sup class="c"><a href="#r14">[14]</a><a href="#r17">[17]</a></sup>. All five are available inside the document that did the work, so the paragraph can be generated instead of remembered.



In [ ]:
methods <- paste(c(
 "Data. Bioactivity data for human ROCK1 (ChEMBL target CHEMBL3231, UniProt Q13464) were retrieved from",
 "ChEMBL release 37 (2026-05-01) restricted to IC50, Ki and Kd measurements. Records without a structure,",
 "with a ChEMBL data-validity flag, without a numeric standard value, or outside the 3-70 heavy-atom window",
 "were removed. Structures were standardised with RDKit (Normalizer, LargestFragmentChooser, Uncharger) and",
 "keyed by InChIKey; measurements of the same parent structure were aggregated by the median of the exact",
 sprintf("(relation '=') values, yielding %d compounds, of which %d have at least one exact measurement and %d",
         nrow(df), nrow(ex), sum(df$censored)),
 "are censored and were excluded from regression. Activities are reported as pActivity = -log10(activity in M).",
 sprintf("The pooled within-compound standard deviation over the %d replicated compounds is %.2f log units,",
         nrow(rep_), sigma_exp),
 sprintf("which we take as the experimental error floor. The curated table (SHA-256 %s...) is deposited with the code.",
         substr(raw_sha, 1, 16)),
 "",
 "Descriptors. Each compound was represented by a Morgan fingerprint (radius 2, 2,048 bits, RDKit",
 sprintf("MorganGenerator) concatenated with %d RDKit physicochemical descriptors (%s),", length(DESC), paste(DESC, collapse = ", ")),
 sprintf("giving %d features. No feature scaling was applied, as the model is tree-based.", ncol(X)),
 "",
 sprintf("Model and validation. A random-forest regressor (R package ranger %s; num.trees = %d, all other",
         as.character(packageVersion("ranger")), HP$num.trees),
 sprintf("hyperparameters at package defaults, seed = %d) was evaluated in two ways: (i) five-fold", HP$seed),
 sprintf("cross-validation over all %d compounds, folds assigned with set.seed(0), and (ii) a Bemis-Murcko", length(y)),
 sprintf("scaffold hold-out in which the largest scaffold families (%d compounds) form the training set and the", length(TR)),
 sprintf("remaining %d compounds the test set, with no scaffold shared between them. Metrics are RMSE, MAE, R2", length(TE)),
 "and Spearman rho; 95% confidence intervals were obtained by bootstrapping the test compounds 2,000 times.",
 "",
 sprintf("Software. R %s, data.table %s, Matrix %s, ranger %s, ggplot2 %s, patchwork %s, knitr %s.",
         paste(R.version$major, R.version$minor, sep = "."), packageVersion("data.table"),
         packageVersion("Matrix"), packageVersion("ranger"), packageVersion("ggplot2"),
         packageVersion("patchwork"), packageVersion("knitr")),
 "All random seeds were fixed at 0. The R Markdown document that produced every number and figure in this",
 "work is available in the repository cited below."), collapse = "\n")
cat(methods)
cat(sprintf("\n\n[%d words]\n", lengths(gregexpr("\\S+", methods))))



Read that paragraph as a referee. Every number in it is a variable from the analysis, so it cannot drift from the results. It names the release of the database, not just the database, and states which rows were excluded and why — censored compounds are the kind of silent exclusion that invalidates comparisons between papers. It gives the hyperparameters that were set and says the rest are package defaults, the only honest way to describe a model with a dozen arguments. And it states a seed, which converts "we obtained" into "anyone obtains".

What it does not do is justify the choices; that belongs in the Results or the Discussion. A Methods section is a recipe, not an argument. One omission is worth repairing by hand: the libraries that did the work should be cited as software, not merely named — R itself<sup class="c"><a href="#r32">[32]</a></sup>, `ranger`<sup class="c"><a href="#r33">[33]</a></sup>, the tidyverse<sup class="c"><a href="#r34">[34]</a></sup> and `ggplot2`<sup class="c"><a href="#r3">[3]</a></sup> all have citable papers, and Unit 24 returns to software citation as an obligation rather than a courtesy.

<div class="exercise"><h4>Exercise 3</h4>
<p>Delete one sentence from the generated Methods and ask a classmate to reproduce the scaffold hold-out RMSE from what remains. Repeat with a different sentence removed. Rank the sentences by how much damage their removal does; the top three are the ones referees ask about, and the exercise is a cheap simulation of peer review.</p></div>

## 12. The reporting checklists the laboratory is held to

Four communities the group publishes in have written down what a paper must report. They overlap, they are short, and filling one in takes twenty minutes — which is the best-value twenty minutes in the whole writing process, because every item a checklist flags is an item a referee would have flagged later.

**OECD principles and the QMRF.** The five OECD principles for the validation of (Q)SAR models<sup class="c"><a href="#r12">[12]</a><a href="#r13">[13]</a></sup> are the practical baseline for regulatory and academic QSAR: a defined endpoint, an unambiguous algorithm, a defined applicability domain, appropriate measures of goodness-of-fit, robustness and predictivity, and — where possible — a mechanistic interpretation. The QMRF (QSAR Model Reporting Format) is the structured document that records them.



In [ ]:
oecd <- data.frame(
  principle = c("1. Defined endpoint", "2. Unambiguous algorithm", "3. Defined applicability domain",
                "4. Goodness-of-fit, robustness, predictivity", "5. Mechanistic interpretation"),
  status = c("met", "met", "partly", "met", "partly"),
  evidence = c(
    sprintf("pActivity = -log10(M) from ChEMBL 37 IC50/Ki/Kd for CHEMBL3231; median over exact values; %d compounds", nrow(ex)),
    sprintf("Morgan(2, 2048) + %d RDKit descriptors; ranger(num.trees = %d, seed = %d), ranger %s",
            length(DESC), HP$num.trees, HP$seed, as.character(packageVersion("ranger"))),
    "k-NN Tanimoto distance to the training set defined in Unit 17; the degradation table is reported there, not here",
    sprintf("random 5-fold CV RMSE %.2f; scaffold hold-out RMSE %.2f [%.2f, %.2f], R2 %.2f; baseline %.2f; experimental floor %.2f",
            rmse_cv, rmse_sc, ci_rmse[2], ci_rmse[3], r2_sc, dummy, sigma_exp),
    "feature attributions available (Unit 16); no causal claim is made about individual substructures"))
kable(oecd, col.names = c("OECD principle", "status", "evidence in this work"))



Two of the five are only partly met, and saying so is the point: a checklist with five green ticks for a notebook-grade model is decoration. The QMRF itself has nine sections — identifier, general information, endpoint, algorithm, applicability domain, internal and external validation, mechanistic interpretation, miscellaneous — and the laboratory keeps one per published model, generated like the Methods paragraph above.

**TRIPOD+AI** applies when the prediction target is a clinical outcome in people<sup class="c"><a href="#r10">[10]</a></sup>. It is the checklist for the group's biomedical projects — the periodontitis expression data of Unit 21, say, if it were used to predict disease status — and it is *not* a QSAR checklist. Students routinely try to apply all four checklists to everything, which produces a long table of "not applicable" entries and hides the handful of items that matter. The chunk below fills in the subset of TRIPOD+AI items that a computational chemistry paper would still be judged on, and marks the rest honestly.

**MIAME and MINSEQE** are the minimum-information standards for microarray and sequencing experiments<sup class="c"><a href="#r11">[11]</a></sup>: raw data, processed data, sample annotation, experimental design, protocol and platform description, deposited in a public repository (GEO, ArrayExpress) with an accession that the paper quotes. **LiveCoMS best practices** play the same role for molecular simulation<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup>: report the force field and its version, the water model, the thermostat and barostat with their coupling constants, the integration time step, the equilibration protocol, the number and length of *independent replicas*, and an uncertainty estimate that accounts for correlation in time.



In [ ]:
tripod <- data.frame(
  item = c("Title and abstract identify the model type and the validation",
           "Source of data and eligibility criteria", "Outcome definition, blinded to predictors",
           "Predictors defined and measured identically in training and use", "Sample size justification",
           "Missing data handling", "Model specification, estimation and all hyperparameters",
           "Performance measures with uncertainty", "Model fairness / subgroup performance",
           "Data and code availability", "Limitations and intended use"),
  status = c("met", "met", "met", "met", "partly", "met", "met", "met", "n/a", "met", "met"),
  note = c("QSAR regression, scaffold hold-out", "ChEMBL 37, CHEMBL3231, IC50/Ki/Kd, human",
           "pActivity from the assay, independent of descriptors", "RDKit, fixed version and parameters",
           sprintf("all available data (%d compounds); no power calculation", nrow(ex)),
           "censored measurements excluded and counted; no imputation", "reported in Methods",
           "RMSE/MAE/R2/rho with bootstrap CIs", "no human subgroups; scaffold families play the analogous role",
           "curated table and document deposited", "stated in the model card (Unit 26)"))
miame <- data.frame(
  component = c("Raw data for each hybridisation/run", "Final processed data",
                "Sample annotation and experimental factors", "Experimental design, replicate relationships",
                "Annotation of the array/platform", "Protocols: sample preparation, labelling, normalisation"),
  how = c("GEO GSE16134 CEL files", "log2 RMA matrix, 5,000 most variable genes x 138 samples",
          "patient_id, status (diseased/healthy), 69 within-patient pairs",
          "paired design; pairing is the analysis unit", "Affymetrix GPL570",
          "RMA; as published by the depositing laboratory"))
livecoms <- data.frame(
  item = c("Force field and version", "Water model, ions and box size",
           "Thermostat / barostat and coupling constants", "Integration time step and constraints",
           "Equilibration protocol and discarded time", "Number and length of independent replicas",
           "Uncertainty accounting for time correlation", "Availability of input files and trajectories"),
  requirement = c("report exactly; a version change is a different model",
                  "report; affects diffusion and electrostatics", "report; a bad thermostat invents dynamics",
                  "report; 2 fs requires constrained bonds to hydrogen",
                  "report; the discarded portion is a decision",
                  "the course dataset: 3 replicas x 1 ns per condition",
                  "block averaging or autocorrelation time, not naive SEM",
                  "deposit inputs; trajectories at least on request"))
cat(sprintf("TRIPOD+AI: %d items scored (%s)\n", nrow(tripod),
            paste(sprintf("%s %d", names(table(tripod$status)), as.integer(table(tripod$status))), collapse = ", ")))
kable(tripod, col.names = c("TRIPOD+AI item (abridged)", "status", "note"))
kable(miame, col.names = c("MIAME / MINSEQE component", "how the course's GEO dataset satisfies it"))
kable(livecoms, col.names = c("LiveCoMS best-practice item", "what must appear in the paper"))



The three tables above are filled in for three different data types, and only the first describes *this* analysis. The second and third are kept in the unit because every student in the group will write one of those papers eventually, and because a filled checklist is easier to adapt than an empty one.



In [ ]:
comp <- rbind(
  data.frame(checklist = sprintf("OECD (%d)", nrow(oecd)), status = oecd$status),
  data.frame(checklist = sprintf("TRIPOD+AI (%d)", nrow(tripod)), status = tripod$status))
comp <- as.data.frame(table(comp$checklist, comp$status)); names(comp) <- c("checklist", "status", "n")
comp$status <- factor(comp$status, levels = c("met", "partly", "n/a"))
print(ggplot(comp[comp$n > 0, ], aes(n, checklist, fill = status)) +
  geom_col(width = 0.45) +
  geom_text(aes(label = n), position = position_stack(vjust = 0.5), colour = "white", size = 2.8) +
  scale_fill_manual(values = c(met = "#009E73", partly = "#E69F00", `n/a` = "#999999")) +
  labs(x = "checklist items", y = NULL, fill = NULL, title = "Reporting compliance of this analysis"))
cat("OECD:", paste(sprintf("%s=%d", names(table(oecd$status)), as.integer(table(oecd$status))), collapse = " "), "\n")
cat("TRIPOD+AI:", paste(sprintf("%s=%d", names(table(tripod$status)), as.integer(table(tripod$status))), collapse = " "), "\n")
cat("MIAME/MINSEQE and LiveCoMS are not scored here: this analysis contains neither omics nor simulation,\n")
cat("and a checklist reported as satisfied for data you do not have is worse than no checklist.\n")



<p class="caption"><b>Figure 9.</b> Items of the two applicable checklists, classified as fully met, partly met or not applicable. The orange segments are the honest part of the figure: an applicability domain defined in another unit rather than here, an absent sample-size justification and an uncommitted mechanistic interpretation are the items this analysis would be asked about in review. The two checklists that do not apply to this data type are left unscored.</p>

## 13. Availability, licences and preprints

Two short statements at the end of a paper do more for its reuse than any amount of Discussion. A **data-availability statement** says where the data are, under what identifier, and what a reader must do to get them; "available from the authors on reasonable request" is, empirically, a statement that the data are not available<sup class="c"><a href="#r20">[20]</a><a href="#r21">[21]</a></sup>. The target is the FAIR standard — findable, accessible, interoperable, reusable<sup class="c"><a href="#r18">[18]</a></sup> — and the incentive is not only ethical: papers with openly deposited data are cited measurably more often<sup class="c"><a href="#r19">[19]</a></sup>. A **code-availability statement** names a repository, a release tag or commit and, ideally, an archival DOI from a service such as Zenodo, because a GitHub repository can be deleted and a DOI cannot.

Licences are where the laboratory's own obligations appear. ChEMBL data are distributed under CC BY-SA 3.0, a *share-alike* licence: a curated table derived from ChEMBL must be redistributed under the same terms, and quietly putting it under CC BY or under no licence at all is a licence violation, not an oversight. Code in the group is released under the MIT licence unless a dependency forces otherwise, and figures under CC BY 4.0.



In [ ]:
avail <- paste(c(
 "Data availability. The curated ROCK1 activity table analysed in this study was derived from ChEMBL release",
 sprintf("37 (target CHEMBL3231) and is deposited with the analysis code (file rock1_chembl_curated.csv, %d rows,", nrow(df)),
 sprintf("SHA-256 %s...). ChEMBL is distributed under CC BY-SA 3.0; the derived table is redistributed under", substr(raw_sha, 1, 24)),
 "the same licence. Supplementary Table S1 contains the per-compound predictions of the hold-out set",
 sprintf("(%d rows, SHA-256 %s...) and its data dictionary.", nrow(supp), substr(sha_s1, 1, 16)),
 "",
 "Code availability. All analyses were performed with the R Markdown document deposited at",
 "https://github.com/<organisation>/<repository> (release v1.0.0, archived at Zenodo under",
 "doi:10.5281/zenodo.<id>), licensed under the MIT licence. The document reproduces every figure and every",
 "number reported here from the deposited data in under five minutes on a laptop.",
 "",
 "Author contributions, competing interests and funding are reported in the sections required by the journal."),
 collapse = "\n")
cat(avail, "\n\n")
lic <- data.frame(
  artefact = c("Source data (ChEMBL)", "Curated table and supplement", "Analysis code",
               "Figures and text of the paper", "Trained model artefacts"),
  licence = c("CC BY-SA 3.0", "CC BY-SA 3.0", "MIT", "CC BY 4.0", "CC BY-SA 3.0"),
  obligation = c("share-alike: derived tables must carry the same licence", "inherited from the source",
                 "permissive; compatible with reuse in commercial settings",
                 "attribution only; required by most open-access journals",
                 "a model trained on share-alike data inherits the obligation"))
kable(lic, col.names = c("artefact", "licence", "what it obliges"))



The statement above contains four things a reuser needs and most statements omit: the release of the source database, the file name, its checksum and its licence. A preprint is the third element. Posting to ChemRxiv, bioRxiv or arXiv at submission establishes priority and makes the work citable while it is in review; almost every journal the laboratory targets permits it, and the exceptions are listed in the journal's policy page, to be checked before posting rather than after.

## 14. The abstract, and the ladder of claims

The abstract is read a hundred times for every time the paper is read, so it is written last and rewritten most. A structured four-move abstract fits any computational paper: what problem and why it matters (two sentences), what was done (two sentences, with the data size), what was found (two or three sentences, with the key numbers and their uncertainty), and what it means and for whom (one sentence, bounded by the applicability domain). The hardest discipline is the last move, because it is where over-claiming lives.

The *ladder of claims* is a device the group uses in internal review. For any result, write the claim at four strengths and choose the highest one the evidence supports: (1) "we observed", (2) "the model ranks", (3) "the model predicts", (4) "the model enables". Evidence of a scaffold-split RMSE somewhat above the experimental noise floor supports rung two comfortably, rung three with the domain stated, and rung four not at all without prospective validation.



In [ ]:
abstract <- paste(c(
 "Rho-associated kinase 1 (ROCK1) is a therapeutic target in glaucoma, fibrosis and cardiovascular disease,",
 "and public bioactivity data for it are large enough to support ligand-based models but heterogeneous enough",
 sprintf("to mislead them. We curated %d compounds with measured ROCK1 affinity from ChEMBL 37 (%d with exact", nrow(df), nrow(ex)),
 sprintf("values) and estimated the experimental error floor of the data at %.2f log units from replicate", sigma_exp),
 sprintf("measurements. A random forest on Morgan fingerprints and physicochemical descriptors reached an RMSE of %.2f", rmse_cv),
 sprintf("log units under random five-fold cross-validation, but %.2f [%.2f, %.2f] on a Bemis-Murcko scaffold hold-out",
         rmse_sc, ci_rmse[2], ci_rmse[3]),
 sprintf("of %d compounds sharing no scaffold with the training set, with R2 %.2f. Selecting the 10 top-ranked", length(TE), r2_sc),
 sprintf("hold-out compounds recovered a potent fraction of %.2f against a base rate of %.2f. The model therefore", hits[1], base_rate),
 "ranks unseen chemotypes usefully while predicting their absolute potency to little better than the measurement",
 "error, and we report it with its applicability domain, its uncertainty and the code that produced it."),
 collapse = "\n")
cat(abstract)
n_words <- lengths(gregexpr("\\S+", abstract))
cat(sprintf("\n\n[%d words | sample sizes present: %s | uncertainty reported: %s | bounded claims: %d]\n",
            n_words, grepl("[0-9]{3,4}", abstract), grepl("\\[", abstract),
            sum(vapply(c("therefore", "while", "little better"),
                       function(w) lengths(gregexpr(w, abstract, fixed = TRUE)), numeric(1)))))
cat("rung of the claim ladder used: 2-3 ('ranks usefully', 'predicts to little better than the error')\n")



That abstract is deliberately unexciting, and it is the version that survives review. It contains the two numbers a referee will look for (the optimistic one and the honest one), it reports an interval, and its final sentence states both what the model does and what it does not do. An abstract claiming that the model "accurately predicts ROCK1 inhibition" would use the same experiments and would be false at rung four.

## 15. Large language models: disclosure, and the reference check you must run

Language models are now part of how most of this material is written and coded, and pretending otherwise helps nobody. The publishers the group submits to have converged on three rules<sup class="c"><a href="#r23">[23]</a><a href="#r24">[24]</a></sup>: a language model cannot be an author, because it cannot take responsibility; its use must be disclosed, usually in the Methods or an Acknowledgements-adjacent statement; and the authors remain fully responsible for every sentence, every number and every citation in the manuscript. Surveys of researchers show the practice is widespread and the disclosure is not<sup class="c"><a href="#r22">[22]</a></sup>.

The laboratory's position is practical. A language model is an acceptable tool for improving the English of a paragraph you wrote, for drafting boilerplate, for explaining an error message and for suggesting code that you then test. It is not an acceptable tool for generating results, for writing a Discussion you have not thought through, or — above all — for producing citations. The characteristic failure mode is a *plausible* reference: correct-looking authors, a real journal, a DOI that is either nonexistent or belongs to a different paper, at rates high enough that an automated check is now part of the group's submission routine<sup class="c"><a href="#r25">[25]</a></sup>.

That check is twenty lines of R. For every DOI in the bibliography, ask Crossref what the title of that DOI is, and compare it with the title in your reference list. The next chunk runs it on a short bibliography into which two bad entries have been planted: one DOI that does not exist, and one real DOI attached to the wrong title. The Python edition used `difflib`; R has no equivalent in base, so the comparison uses a normalised Levenshtein distance from `utils::adist()`, which gives a slightly different similarity for the same pair of strings but the same verdicts at the same threshold.



In [ ]:
BIB <- list(
  c("10.1021/ci400197w", "Modeling, informatics, and the quest for reproducibility"),
  c("10.1371/journal.pcbi.1005619", "Ten simple rules for structuring papers"),
  c("10.1038/s41592-021-01256-7", "Reproducibility standards for machine learning in the life sciences"),
  c("10.1136/bmj-2023-078378", paste("TRIPOD+AI statement: updated guidance for reporting clinical prediction",
                                     "models that use regression or machine learning methods")),
  c("10.1021/jm300131x", "A deep learning framework for ROCK1 inhibitor design"),      ## planted: wrong title
  c("10.1038/s41586-024-99999-9", "Graph neural networks predict kinase selectivity")) ## planted: no such DOI
FROZEN <- list("10.1021/ci400197w" = "Modeling, Informatics, and the Quest for Reproducibility",
               "10.1371/journal.pcbi.1005619" = "Ten simple rules for structuring papers",
               "10.1038/s41592-021-01256-7" = "Reproducibility standards for machine learning in the life sciences",
               "10.1136/bmj-2023-078378" = paste("TRIPOD+AI statement: updated guidance for reporting clinical",
                                                 "prediction models that use regression or machine learning methods"),
               "10.1021/jm300131x" = "The Experimental Uncertainty of Heterogeneous Public Ki Data",
               "10.1038/s41586-024-99999-9" = NA_character_)
normalise <- function(t) {
  t <- gsub("<[^>]+>", " ", tolower(ifelse(is.na(t), "", t)))
  gsub("\\s+", " ", trimws(gsub("[^a-z0-9 ]", " ", t)))
}
crossref_title <- function(doi) {
  resp <- request("https://api.crossref.org/works/") |> req_url_path_append(doi) |>
    req_user_agent("LBQC course material (reference integrity check)") |>
    req_timeout(12) |> req_perform()
  resp_body_json(resp)$message$title[[1]]
}
live <- tryCatch({ crossref_title(BIB[[1]][1]); TRUE }, error = function(e) FALSE)
rows <- lapply(BIB, function(bb) {
  found <- if (live) tryCatch(crossref_title(bb[1]), error = function(e) NA_character_) else FROZEN[[bb[1]]]
  a <- normalise(bb[2]); b <- normalise(found)
  sim <- if (is.na(found)) 0 else 1 - as.numeric(adist(a, b)) / max(nchar(a), nchar(b))
  data.frame(doi = bb[1], similarity = round(sim, 2),
             verdict = if (is.na(found)) "DOI NOT FOUND" else if (sim >= 0.75) "ok" else "TITLE MISMATCH",
             registered_title = substr(ifelse(is.na(found), "-", found), 1, 46))
})
check <- do.call(rbind, rows)
cat(sprintf("path taken: %s\n", if (live) "live Crossref queries" else "frozen fallback list"))
kable(check)
cat(sprintf("flagged %d of %d entries\n", sum(check$verdict != "ok"), nrow(check)))


In [ ]:
check$lab <- sprintf("%s  [%s]", check$doi, check$verdict)
check$lab <- factor(check$lab, levels = rev(check$lab))
print(ggplot(check, aes(similarity, lab, fill = verdict == "ok")) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_vline(xintercept = 0.75, linetype = 2, linewidth = 0.4) +
  annotate("text", x = 0.76, y = 6.4, hjust = 0, size = 2.4, label = "acceptance threshold 0.75") +
  scale_fill_manual(values = c(`TRUE` = "#009E73", `FALSE` = "#D55E00")) +
  xlim(0, 1.25) + labs(x = "title similarity to the registered record", y = NULL,
                       title = "Reference-integrity check of a six-entry bibliography") +
  theme(axis.text.y = element_text(size = 6)))
cat("entries below the threshold:", paste(check$doi[check$verdict != "ok"], collapse = ", "), "\n")



<p class="caption"><b>Figure 10.</b> Title similarity between each bibliography entry and the record registered for its DOI. Four entries match closely; the fifth is a real DOI (a 2012 paper on the experimental uncertainty of public affinity data) carrying a fabricated title, and the sixth is a DOI that was never registered. Both planted errors are caught, and neither would have been caught by reading the reference list.</p>

The threshold of 0.75 is a choice, not a law: subtitles, non-ASCII characters and markup in the registered record depress the similarity of good entries, which is why the check *flags* rather than deletes — and why the character-level measure used here is more conservative than the token-level one of the Python edition for titles that differ only in capitalisation or punctuation. Run it before every submission; it catches errors copied from a slide deck or a colleague's manuscript as readily as those from a language model.

A worked disclosure statement, in the form the laboratory uses:

<div class="box"><h4>Statement on the use of generative tools</h4>
<p>During the preparation of this work the authors used a large language model to improve the readability and the English style of several paragraphs of the Introduction and Discussion, and to draft documentation comments in the analysis code. All text was subsequently reviewed and edited by the authors, every reference was verified against its registered Crossref record, and all code was tested against the unit tests in the deposited repository. The authors take full responsibility for the content of the publication.</p></div>

## 16. Summary

**The last mile is computation, not decoration.** Figures, tables, the Methods paragraph, the availability statements, the checklists and even the abstract were generated here from the analysis that produced the results, and the ones that are generated stay correct when the results change.

**Figures are physical objects.** Draw at the final width in centimetres with `ggsave(units = "cm")`, set the font size once in a theme function, export vector with `cairo_pdf` so the fonts are embedded — the default `pdf()` device does not embed them — and check the figure under colour-vision deficiency and in greyscale before it is submitted<sup class="c"><a href="#r1">[1]</a><a href="#r6">[6]</a><a href="#r8">[8]</a></sup>. The remake in Section 5 changed nothing about the data and everything about what a reader can take from it.

**The misleading figure is the dangerous one.** A truncated axis turned a difference of a fraction of a log unit — smaller than the experimental noise, with a confidence interval containing zero — into a visual factor of several, and a bar of means hid two almost completely overlapping distributions<sup class="c"><a href="#r5">[5]</a></sup>. Bars start at zero; means travel with their distributions; differences are plotted against the noise floor.

**Digits are claims.** Round the uncertainty first, then the value; report an error in the units of the target, a rank metric, the sample size and the split; and never let a single flattering metric stand for the model's performance<sup class="c"><a href="#r2">[2]</a></sup>.

**Checklists are cheap insurance.** OECD and QMRF for QSAR<sup class="c"><a href="#r12">[12]</a><a href="#r13">[13]</a></sup>, TRIPOD+AI for clinical prediction<sup class="c"><a href="#r10">[10]</a></sup>, MIAME and MINSEQE for omics<sup class="c"><a href="#r11">[11]</a></sup>, LiveCoMS for simulation<sup class="c"><a href="#r15">[15]</a><a href="#r16">[16]</a></sup>: fill in the applicable one, mark partly-met items honestly, leave the rest unscored.

**Disclose the tools and verify the references.** A language model is a writing and coding assistant whose use is disclosed and whose output is checked; the reference-integrity check of Section 15 is twenty lines of `httr2` and catches the one error that most damages a manuscript's credibility<sup class="c"><a href="#r25">[25]</a></sup>.

What follows this unit is the machinery that makes the claims checkable by someone else: <a href="../U24_Software_Engineering/index.html">Unit 24</a> turns the analysis code into a tested R package, <a href="../U25_Scaling_HPC/index.html">Unit 25</a> into a workflow that runs on a cluster, and <a href="../U26_Apps_Deployment/index.html">Unit 26</a> into a tool a collaborator can use — each with its own form of documentation, and each pointing back at the figures and numbers assembled here.

<div class="box bridge"><h4>Bridge</h4>
<p>The evidence communicated here was built earlier in this course and in its sister courses. The exploratory figure remade in Section 5 comes from <a href="../U5_EDA/index.html">Unit 5</a>; the experimental error floor and the data card from <a href="../U6_Data_Quality/index.html">Unit 6</a>; the scaffold split and the honest metric from <a href="../U12_Validation/index.html">Unit 12</a>; the model from <a href="../U14_Ensembles_Tuning/index.html">Unit 14</a>; the applicability domain and the OECD discussion from <a href="../U17_Uncertainty_AD/index.html">Unit 17</a>; the enrichment panel from the virtual-screening unit, <a href="../U19_Virtual_Screening/index.html">U19 of the Python edition</a>; the simulation reporting items from <a href="../U20_MD_Analytics/index.html">U20</a> and the omics ones from <a href="../U21_Omics/index.html">U21</a>. In the cheminformatics course, <a href="../../../Cheminformatics/U7_Data_Curation/index.html">Data curation</a> and <a href="../../../Cheminformatics/U14_Validation/index.html">Validation</a> cover the chemistry-specific reporting that referees at cheminformatics journals expect, and <a href="../../../Cheminformatics/U12_Classical_QSAR/index.html">Classical QSAR</a> the QMRF tradition. In the bioinformatics course, <a href="../../../BGM324/05_Molecular_Dynamics/index.html">Molecular dynamics</a> supplies the simulation protocol details that LiveCoMS asks you to report, and in the R edition of the AI course <a href="../../../BGM411/R/S1_Model_Evaluation/index.html">Model evaluation</a> covers the metric definitions quoted in the tables above. The Python edition of this unit is <a href="../../Python/U23_Communication/index.html">here</a>.</p></div>

## 17. Self-assessment

<details><summary>1. A journal asks for figures at 300 dpi, maximum width 8.5 cm. You drew your figure with the device defaults. What exactly goes wrong, and what is the fix?</summary>
<p>R's default device is 7 inches = 17.8 cm wide, so the typesetter scales it by about 0.48. Every font, line width and marker shrinks by the same factor: a 9-point label arrives at about 4.3 points, below the publisher's minimum, and hairlines disappear. Enlarging the fonts afterwards changes the layout. The fix is <code>ggsave(file, plot, width = 8.5, height = h, units = "cm")</code>, choosing the base font size at that size with <code>theme_lbqc()</code>, and exporting vector (PDF) so the dpi requirement applies only to raster panels.</p></details>

<details><summary>2. Why is a PDF usually smaller than a 300 dpi PNG of the same scatter plot, and when is the reverse true?</summary>
<p>A PDF stores one drawing instruction per mark, a PNG stores every pixel of the canvas. With a few thousand marks the instructions win, as measured in Section 2. The reverse is true once the number of marks approaches the number of pixels — tens of thousands of points, a dense heat map, a rendered molecular surface — which is when rasterising the heavy data layer while keeping the axes and text vector is the right compromise.</p></details>

<details><summary>3. In R, which device do you use to export a figure with embedded fonts, and how do you check?</summary>
<p><code>cairo_pdf</code> (or <code>ggsave(..., device = cairo_pdf)</code>); the default <code>pdf()</code> device references the base-14 fonts by name instead of embedding them. The check is three lines: read the file as raw bytes and search for the string <code>/FontFile</code>, as the first chunk of Section 2 does — it is TRUE for the cairo file and FALSE for the default one.</p></details>

<details><summary>4. Your supervisor says the two scaffold families in Figure 5 "clearly differ in potency". What do you answer, with numbers?</summary>
<p>The difference of means is a fraction of a log unit, its bootstrap confidence interval includes zero, Cohen's d is small, and the experimental standard deviation of a single measurement is several times the difference itself. The honest statement is that the two families are indistinguishable at the precision of the data; the truncated-axis bar chart created the impression of a difference by removing the zero and the spread.</p></details>

<details><summary>5. Write the reported value for a metric of 0.73819 with a 95 % interval of [0.70, 0.78].</summary>
<p>The half-width is about 0.04, one significant figure, two decimals; so the value is reported as 0.74 with the interval [0.70, 0.78]. Reporting 0.73819 claims five significant figures for a quantity known to two. Had the half-width been 0.012, two significant figures would be kept and the value reported as 0.738 — the rule implemented in <code>round_to_uncertainty()</code>.</p></details>

<details><summary>6. Which number belongs in the abstract: the random-split RMSE or the scaffold-split RMSE?</summary>
<p>Both, with the claim attached to the scaffold one. Quoting only the random-split value answers a question nobody asked (how well the model interpolates among chemotypes it has already seen) and is the single most common over-claim in ligand-based modelling papers. The pair also tells the reader how much of the apparent performance was scaffold memorisation — about a sixth of the headline RMSE here.</p></details>

<details><summary>7. Your model predicts a compound's potency. Do you fill in TRIPOD+AI?</summary>
<p>No, or only the items that generalise. TRIPOD+AI is written for prediction models whose outcome is a clinical event in people, and most of its items — participants, eligibility, subgroup fairness, calibration in clinical terms — have no referent in a QSAR study. The applicable checklist is the OECD set with a QMRF; TRIPOD+AI becomes the right one the moment the group's biomedical projects predict a patient outcome. Filling in every checklist for every paper produces long tables of "not applicable" and hides the items that matter.</p></details>

<details><summary>8. What must a data-availability statement contain to be useful?</summary>
<p>The identity of the source (database, release, accession or query), the location of the derived data (repository with a persistent identifier), the licence, and the checksum or row count that lets a reader confirm they have the same file. "Available from the authors on reasonable request" fails all four and is empirically equivalent to unavailable.</p></details>

<details><summary>9. The data came from ChEMBL. What licence can your curated table carry?</summary>
<p>CC BY-SA 3.0, the same as the source, because share-alike propagates to derivative works. The code that produced the table can still be MIT: the licence of the data and the licence of the software are separate. A trained model whose parameters encode the data is treated by the group as a derivative of the data.</p></details>

<details><summary>10. Describe the reference-integrity check in three steps and state its two failure modes.</summary>
<p>For each DOI in the bibliography: request <code>https://api.crossref.org/works/&lt;doi&gt;</code> with <code>httr2</code>, read the registered title, compare it with the cited title by string similarity, and flag entries below a threshold. It fails in two directions: false alarms, when the registered title contains markup, a subtitle or non-ASCII characters; and misses, when a reference has no DOI at all, which is the case for books, theses and some conference papers — those still require a human check.</p></details>

<details><summary>11. What may a language model legitimately do in the preparation of a paper from this laboratory, and what must be disclosed?</summary>
<p>It may improve the English of text the authors wrote, draft boilerplate such as availability statements and documentation comments, explain errors and suggest code that is then tested. It may not generate results, write an argument the authors have not made, or supply citations. Its use is disclosed in a short statement naming what it was used for, the authors verify every reference and every number, and authorship remains with the humans who take responsibility.</p></details>

<details><summary>12. Why does the laboratory write the figure captions before the Results section?</summary>
<p>Because a caption forces the claim to point at visible evidence. If the caption cannot be written in two or three sentences that a reader who skipped the text would understand, the panel is not finished — and the claims that turn out to point at nothing are deleted while deleting them is still cheap. It also fixes the order of the figures, which fixes the order of the argument.</p></details>

## Glossary

<dl class="gloss">
<dt>Applicability domain</dt><dd>The region of chemical or feature space in which a model's predictions are considered reliable, stated explicitly and supported by evidence of degradation outside it.</dd>
<dt>Bootstrap confidence interval</dt><dd>Interval obtained by resampling the test cases with replacement and taking percentiles of the recomputed metric.</dd>
<dt>Caption contract</dt><dd>The requirement that a caption let a reader who skipped the text understand the panel, name the statistical objects shown, give the sample size, and assert nothing the panel does not show.</dd>
<dt>cairo_pdf</dt><dd>The R graphics device that writes PDF with the glyphs embedded; the default <code>pdf()</code> device references fonts by name instead.</dd>
<dt>Data-availability statement</dt><dd>The sentence naming the source, location, licence and identifier of the data behind a paper.</dd>
<dt>Deuteranopia</dt><dd>The most common colour-vision deficiency, a reduced ability to distinguish red from green; simulated here by a linear transformation of sRGB values.</dd>
<dt>Figure ordering</dt><dd>The convention that figures run data, method, main result, control, application, and that panels are read left to right as an argument.</dd>
<dt>LiveCoMS best practices</dt><dd>Community articles specifying what a molecular-simulation paper must report: force field, water model, thermostat, time step, equilibration, replicas and correlation-aware uncertainty.</dd>
<dt>MIAME / MINSEQE</dt><dd>Minimum-information standards for microarray and sequencing experiments: raw data, processed data, sample annotation, design, platform and protocols, deposited with an accession.</dd>
<dt>Okabe-Ito palette</dt><dd>An eight-colour categorical palette that remains distinguishable under the common colour-vision deficiencies.</dd>
<dt>patchwork</dt><dd>The R package that composes complete ggplot2 objects into multi-panel figures with the operators <code>|</code> and <code>/</code> and adds panel letters with <code>plot_annotation(tag_levels)</code>.</dd>
<dt>Perceptually uniform colour map</dt><dd>A sequential map (viridis, cividis, magma) whose luminance increases monotonically, so equal steps in value look like equal steps in colour and the map survives greyscale printing.</dd>
<dt>QMRF</dt><dd>QSAR Model Reporting Format: the structured document whose nine sections record a model against the five OECD validation principles.</dd>
<dt>Significant figures tied to uncertainty</dt><dd>The convention of rounding the uncertainty to one significant figure (two if it begins with 1) and the value to the same decimal place.</dd>
<dt>Theme (ggplot2)</dt><dd>The function holding every non-data appearance setting of a plot; installed globally with <code>theme_set()</code> and shipped with the manuscript as an R file.</dd>
<dt>TRIPOD+AI</dt><dd>The 2024 reporting guideline for prediction models of clinical outcomes developed with regression or machine learning.</dd>
<dt>Vector vs raster</dt><dd>Formats that store drawing instructions (PDF, SVG, EPS) versus pixels (PNG, TIFF); vector is resolution-independent and preferred for plots with a moderate number of marks.</dd>
</dl>



In [ ]:
produced <- sort(basename(list.files(WORK)))
cat("files produced by this document:", paste(produced, collapse = ", "), "\n")
unlink(WORK, recursive = TRUE)
cat("scratch folder removed:", !dir.exists(WORK), "\n")



Nothing a document writes outside its own outputs should survive it: the figures, the style file, the supplement and its dictionary were written into the session's `tempdir()` and are deleted above. In a real manuscript repository the same objects are written into a tracked `figures/` and `tables/` folder instead, and the path appears once, at the top of the document.

## References and further reading

<ol class="refs">
<li id="r1">Rougier NP, Droettboom M, Bourne PE. Ten simple rules for better figures. <em>PLoS Comput Biol</em>. 2014;10(9):e1003833. <a href="https://doi.org/10.1371/journal.pcbi.1003833">doi:10.1371/journal.pcbi.1003833</a></li>
<li id="r2">Cleveland WS, McGill R. Graphical perception: theory, experimentation, and application to the development of graphical methods. <em>J Am Stat Assoc</em>. 1984;79(387):531-554. <a href="https://doi.org/10.1080/01621459.1984.10478080">doi:10.1080/01621459.1984.10478080</a></li>
<li id="r3">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r4">Wilkinson L. <em>The Grammar of Graphics</em>. 2nd ed. Springer; 2005. <a href="https://doi.org/10.1007/0-387-28695-0">doi:10.1007/0-387-28695-0</a></li>
<li id="r5">Weissgerber TL, Milic NM, Winham SJ, Garovic VD. Beyond bar and line graphs: time for a new data presentation paradigm. <em>PLoS Biol</em>. 2015;13(4):e1002128. <a href="https://doi.org/10.1371/journal.pbio.1002128">doi:10.1371/journal.pbio.1002128</a></li>
<li id="r6">Crameri F, Shephard GE, Heron PJ. The misuse of colour in science communication. <em>Nat Commun</em>. 2020;11:5444. <a href="https://doi.org/10.1038/s41467-020-19160-7">doi:10.1038/s41467-020-19160-7</a></li>
<li id="r7">Nunez JR, Anderton CR, Renslow RS. Optimizing colormaps with consideration for color vision deficiency to enable accurate interpretation of scientific data. <em>PLoS ONE</em>. 2018;13(7):e0199239. <a href="https://doi.org/10.1371/journal.pone.0199239">doi:10.1371/journal.pone.0199239</a></li>
<li id="r8">Wong B. Points of view: color blindness. <em>Nat Methods</em>. 2011;8:441. <a href="https://doi.org/10.1038/nmeth.1618">doi:10.1038/nmeth.1618</a></li>
<li id="r9">Mensh B, Kording K. Ten simple rules for structuring papers. <em>PLoS Comput Biol</em>. 2017;13(9):e1005619. <a href="https://doi.org/10.1371/journal.pcbi.1005619">doi:10.1371/journal.pcbi.1005619</a></li>
<li id="r10">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r11">Brazma A, Hingamp P, Quackenbush J, et al. Minimum information about a microarray experiment (MIAME) - toward standards for microarray data. <em>Nat Genet</em>. 2001;29:365-371. <a href="https://doi.org/10.1038/ng1201-365">doi:10.1038/ng1201-365</a></li>
<li id="r12">Gramatica P. Principles of QSAR models validation: internal and external. <em>QSAR Comb Sci</em>. 2007;26(5):694-701. <a href="https://doi.org/10.1002/qsar.200610151">doi:10.1002/qsar.200610151</a></li>
<li id="r13">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r14">Walters WP. Modeling, informatics, and the quest for reproducibility. <em>J Chem Inf Model</em>. 2013;53(7):1529-1530. <a href="https://doi.org/10.1021/ci400197w">doi:10.1021/ci400197w</a></li>
<li id="r15">Grossfield A, Patrone PN, Roe DR, Schultz AJ, Siderius DW, Zuckerman DM. Best practices for quantification of uncertainty and sampling quality in molecular simulations. <em>Living J Comput Mol Sci</em>. 2019;1(1):5067. <a href="https://doi.org/10.33011/livecoms.1.1.5067">doi:10.33011/livecoms.1.1.5067</a></li>
<li id="r16">Braun E, Gilmer J, Mayes HB, et al. Best practices for foundations in molecular simulations. <em>Living J Comput Mol Sci</em>. 2018;1(1):5957. <a href="https://doi.org/10.33011/livecoms.1.1.5957">doi:10.33011/livecoms.1.1.5957</a></li>
<li id="r17">Heil BJ, Hoffman MM, Markowetz F, Lee SI, Greene CS, Hicks SC. Reproducibility standards for machine learning in the life sciences. <em>Nat Methods</em>. 2021;18:1132-1135. <a href="https://doi.org/10.1038/s41592-021-01256-7">doi:10.1038/s41592-021-01256-7</a></li>
<li id="r18">Wilkinson MD, Dumontier M, Aalbersberg IJ, et al. The FAIR guiding principles for scientific data management and stewardship. <em>Sci Data</em>. 2016;3:160018. <a href="https://doi.org/10.1038/sdata.2016.18">doi:10.1038/sdata.2016.18</a></li>
<li id="r19">Piwowar HA, Vision TJ. Data reuse and the open data citation advantage. <em>PeerJ</em>. 2013;1:e175. <a href="https://doi.org/10.7717/peerj.175">doi:10.7717/peerj.175</a></li>
<li id="r20">Stodden V, Seiler J, Ma Z. An empirical analysis of journal policy effectiveness for computational reproducibility. <em>Proc Natl Acad Sci USA</em>. 2018;115(11):2584-2589. <a href="https://doi.org/10.1073/pnas.1708290115">doi:10.1073/pnas.1708290115</a></li>
<li id="r21">Nosek BA, Alter G, Banks GC, et al. Promoting an open research culture. <em>Science</em>. 2015;348(6242):1422-1425. <a href="https://doi.org/10.1126/science.aab2374">doi:10.1126/science.aab2374</a></li>
<li id="r22">Van Noorden R, Perkel JM. AI and science: what 1,600 researchers think. <em>Nature</em>. 2023;621:672-675. <a href="https://doi.org/10.1038/d41586-023-02980-0">doi:10.1038/d41586-023-02980-0</a></li>
<li id="r23">Stokel-Walker C, Van Noorden R. What ChatGPT and generative AI mean for science. <em>Nature</em>. 2023;614:214-216. <a href="https://doi.org/10.1038/d41586-023-00340-6">doi:10.1038/d41586-023-00340-6</a></li>
<li id="r24">Nature editorial. Tools such as ChatGPT threaten transparent science; here are our ground rules for their use. <em>Nature</em>. 2023;613:612. <a href="https://doi.org/10.1038/d41586-023-00191-1">doi:10.1038/d41586-023-00191-1</a></li>
<li id="r25">Walters WH, Wilder EI. Fabrication and errors in the bibliographic citations generated by ChatGPT. <em>Sci Rep</em>. 2023;13:14045. <a href="https://doi.org/10.1038/s41598-023-41032-5">doi:10.1038/s41598-023-41032-5</a></li>
<li id="r26">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r27">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45:5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r28">Ziemann M, Eren Y, El-Osta A. Gene name errors are widespread in the scientific literature. <em>Genome Biol</em>. 2016;17:177. <a href="https://doi.org/10.1186/s13059-016-1044-7">doi:10.1186/s13059-016-1044-7</a></li>
<li id="r29">Baker M. 1,500 scientists lift the lid on reproducibility. <em>Nature</em>. 2016;533:452-454. <a href="https://doi.org/10.1038/533452a">doi:10.1038/533452a</a></li>
<li id="r30">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <em>J Med Chem</em>. 2012;55(11):5165-5173. <a href="https://doi.org/10.1021/jm300131x">doi:10.1021/jm300131x</a></li>
<li id="r31">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r32">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r33">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r34">Wickham H, Averick M, Bryan J, et al. Welcome to the tidyverse. <em>J Open Source Softw</em>. 2019;4(43):1686. <a href="https://doi.org/10.21105/joss.01686">doi:10.21105/joss.01686</a></li>
<li id="r35">Xie Y. <em>Dynamic Documents with R and knitr</em>. 2nd ed. Chapman and Hall/CRC; 2015. <a href="https://doi.org/10.1201/9781315382487">doi:10.1201/9781315382487</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Tufte ER. <em>The Visual Display of Quantitative Information</em>. 2nd ed. Graphics Press; 2001. - The source of the data-ink argument and of the vocabulary for discussing chartjunk.</li>
<li>Healy K. <em>Data Visualization: A Practical Introduction</em>. Princeton University Press; 2018 (free online). - Why a plot works, with the perceptual evidence, and every example in ggplot2.</li>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023 (free online). - Chapters on communication and on graphics for export.</li>
<li>Schimel J. <em>Writing Science</em>. Oxford University Press; 2012. - The narrative structure of a scientific argument, at paragraph resolution.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>OECD. <em>Guidance Document on the Validation of (Quantitative) Structure-Activity Relationship [(Q)SAR] Models</em>, ENV/JM/MONO(2007)2, and the JRC QMRF template: <a href="https://www.oecd.org/chemicalsafety/risk-assessment/validationofqsarmodels.htm">oecd.org</a>.</li>
<li>ICMJE. <em>Recommendations for the Conduct, Reporting, Editing, and Publication of Scholarly Work in Medical Journals</em>, section on artificial-intelligence-assisted technology: <a href="https://www.icmje.org/recommendations/">icmje.org/recommendations</a>.</li>
<li>EQUATOR Network, the index of reporting guidelines including TRIPOD+AI, STARD and CONSORT: <a href="https://www.equator-network.org/">equator-network.org</a>.</li>
<li><code>patchwork</code> reference and the guide to composing and tagging panels: <a href="https://patchwork.data-imaginist.com/">patchwork.data-imaginist.com</a>.</li>
<li>Crossref REST API documentation, used by the reference-integrity check: <a href="https://api.crossref.org/swagger-ui/index.html">api.crossref.org</a>.</li>
<li>The Turing Way, chapters on research communication and open-access publishing: <a href="https://the-turing-way.netlify.app/">the-turing-way.netlify.app</a>.</li>
</ul>
